In [ ]:
import torch
import transformers
import pandas
import numpy

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

print("Environment ready!")


PyTorch: 2.11.0+cpu
Transformers: 5.22.0
CUDA available: False
Running on CPU
Environment ready!


In [ ]:
%pip install -q faker presidio-analyzer presidio-anonymizer seqeval scikit-learn pandas numpy matplotlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
import transformers
import pandas as pd
from faker import Faker
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from seqeval.metrics import classification_report

print("All project dependencies are ready!")

All project dependencies are ready!


In [ ]:
clinical_notes = [
    {
        "id": "NOTE_001",
        "text": "Patient John Miller, aged 45, visited Salem City Hospital on 12/03/2025. Contact: 9876543210. Email: john.miller@example.com. Diagnosed with diabetes."
    },
    {
        "id": "NOTE_002",
        "text": "Dr. Robert Smith examined Mary Johnson on 15/04/2025. Her medical record number is MRN123456. She reported chest pain for 3 days."
    },
    {
        "id": "NOTE_003",
        "text": "Patient David Brown, aged 92, lives at 25 Lake Road, Salem, Tamil Nadu 636001. He was diagnosed with hypertension."
    }
]

for note in clinical_notes:
    print(f"\n{note['id']}:")
    print(note["text"])


NOTE_001:
Patient John Miller, aged 45, visited Salem City Hospital on 12/03/2025. Contact: 9876543210. Email: john.miller@example.com. Diagnosed with diabetes.

NOTE_002:
Dr. Robert Smith examined Mary Johnson on 15/04/2025. Her medical record number is MRN123456. She reported chest pain for 3 days.

NOTE_003:
Patient David Brown, aged 92, lives at 25 Lake Road, Salem, Tamil Nadu 636001. He was diagnosed with hypertension.


In [ ]:
PHI_LABELS = [
    "O",
    "B-NAME", "I-NAME",
    "B-LOCATION", "I-LOCATION",
    "B-DATE", "I-DATE",
    "B-AGE", "I-AGE",
    "B-PHONE", "I-PHONE",
    "B-EMAIL", "I-EMAIL",
    "B-SSN", "I-SSN",
    "B-MRN", "I-MRN",
    "B-HEALTH_PLAN_ID", "I-HEALTH_PLAN_ID",
    "B-ACCOUNT_ID", "I-ACCOUNT_ID",
    "B-LICENSE_ID", "I-LICENSE_ID",
    "B-URL", "I-URL",
    "B-IP", "I-IP"
]

label2id = {label: idx for idx, label in enumerate(PHI_LABELS)}
id2label = {idx: label for label, idx in label2id.items()}

print("Total labels:", len(PHI_LABELS))
print("Label mapping:", label2id)

Total labels: 27
Label mapping: {'O': 0, 'B-NAME': 1, 'I-NAME': 2, 'B-LOCATION': 3, 'I-LOCATION': 4, 'B-DATE': 5, 'I-DATE': 6, 'B-AGE': 7, 'I-AGE': 8, 'B-PHONE': 9, 'I-PHONE': 10, 'B-EMAIL': 11, 'I-EMAIL': 12, 'B-SSN': 13, 'I-SSN': 14, 'B-MRN': 15, 'I-MRN': 16, 'B-HEALTH_PLAN_ID': 17, 'I-HEALTH_PLAN_ID': 18, 'B-ACCOUNT_ID': 19, 'I-ACCOUNT_ID': 20, 'B-LICENSE_ID': 21, 'I-LICENSE_ID': 22, 'B-URL': 23, 'I-URL': 24, 'B-IP': 25, 'I-IP': 26}


In [ ]:
training_examples = [
    {
        "text": "Patient John Miller, aged 45, visited Salem City Hospital on 12/03/2025. Contact: 9876543210. Email: john.miller@example.com. Diagnosed with diabetes.",
        "entities": [
            ("John Miller", "NAME"),
            ("Salem City Hospital", "LOCATION"),
            ("12/03/2025", "DATE"),
            ("9876543210", "PHONE"),
            ("john.miller@example.com", "EMAIL")
        ]
    },
    {
        "text": "Dr. Robert Smith examined Mary Johnson on 15/04/2025. Her medical record number is MRN123456. She reported chest pain for 3 days.",
        "entities": [
            ("Robert Smith", "NAME"),
            ("Mary Johnson", "NAME"),
            ("15/04/2025", "DATE"),
            ("MRN123456", "MRN")
        ]
    },
    {
        "text": "Patient David Brown, aged 92, lives at 25 Lake Road, Salem, Tamil Nadu 636001. He was diagnosed with hypertension.",
        "entities": [
            ("David Brown", "NAME"),
            ("92", "AGE"),
            ("25 Lake Road", "LOCATION"),
            ("Salem", "LOCATION"),
            ("636001", "LOCATION")
        ]
    }
]

def create_annotations(example):
    text = example["text"]
    annotations = []

    for value, label in example["entities"]:
        start = text.find(value)

        if start == -1:
            raise ValueError(f"Entity not found: {value}")

        end = start + len(value)

        annotations.append({
            "start": start,
            "end": end,
            "label": label,
            "text": value
        })

    return {
        "text": text,
        "annotations": annotations
    }

annotated_data = [
    create_annotations(example)
    for example in training_examples
]

for item in annotated_data:
    print("\nClinical note:", item["text"])
    print("Annotations:", item["annotations"])

print("\nTotal annotated notes:", len(annotated_data))


Clinical note: Patient John Miller, aged 45, visited Salem City Hospital on 12/03/2025. Contact: 9876543210. Email: john.miller@example.com. Diagnosed with diabetes.
Annotations: [{'start': 8, 'end': 19, 'label': 'NAME', 'text': 'John Miller'}, {'start': 38, 'end': 57, 'label': 'LOCATION', 'text': 'Salem City Hospital'}, {'start': 61, 'end': 71, 'label': 'DATE', 'text': '12/03/2025'}, {'start': 82, 'end': 92, 'label': 'PHONE', 'text': '9876543210'}, {'start': 101, 'end': 124, 'label': 'EMAIL', 'text': 'john.miller@example.com'}]

Clinical note: Dr. Robert Smith examined Mary Johnson on 15/04/2025. Her medical record number is MRN123456. She reported chest pain for 3 days.
Annotations: [{'start': 4, 'end': 16, 'label': 'NAME', 'text': 'Robert Smith'}, {'start': 26, 'end': 38, 'label': 'NAME', 'text': 'Mary Johnson'}, {'start': 42, 'end': 52, 'label': 'DATE', 'text': '15/04/2025'}, {'start': 83, 'end': 92, 'label': 'MRN', 'text': 'MRN123456'}]

Clinical note: Patient David Brown, aged 9

In [ ]:
for item in annotated_data:
    for entity in item["annotations"]:
        extracted = item["text"][entity["start"]:entity["end"]]

        assert extracted == entity["text"], (
            f"Mismatch! Expected: {entity['text']}, Found: {extracted}"
        )

print("All annotation offsets verified successfully!")
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded:", MODEL_NAME)

All annotation offsets verified successfully!
Tokenizer loaded: distilbert-base-uncased


In [ ]:
def tokenize_and_align(example):
    text = example["text"]

    encoding = tokenizer(
        text,
        truncation=True,
        return_offsets_mapping=True
    )

    token_labels = []

    for token_start, token_end in encoding["offset_mapping"]:

        # Special tokens have (0, 0) offsets
        if token_start == token_end:
            token_labels.append(-100)
            continue

        assigned_label = "O"

        for entity in example["annotations"]:
            entity_start = entity["start"]
            entity_end = entity["end"]

            # Check whether this token overlaps an entity
            if token_start < entity_end and token_end > entity_start:

                if token_start == entity_start:
                    assigned_label = "B-" + entity["label"]
                else:
                    assigned_label = "I-" + entity["label"]

                break

        token_labels.append(label2id[assigned_label])

    encoding["labels"] = token_labels

    return encoding


# Align all three training examples
tokenized_data = [
    tokenize_and_align(item)
    for item in annotated_data
]

print("BIO token alignment completed!")

# Display the first note's tokens and labels
first_tokens = tokenizer.convert_ids_to_tokens(
    tokenized_data[0]["input_ids"]
)

first_labels = tokenized_data[0]["labels"]

for token, label_id in zip(first_tokens, first_labels):
    label = "SPECIAL TOKEN" if label_id == -100 else id2label[label_id]
    print(f"{token:25} {label}")

BIO token alignment completed!
[CLS]                     SPECIAL TOKEN
patient                   O
john                      B-NAME
miller                    I-NAME
,                         O
aged                      O
45                        O
,                         O
visited                   O
salem                     B-LOCATION
city                      I-LOCATION
hospital                  I-LOCATION
on                        O
12                        B-DATE
/                         I-DATE
03                        I-DATE
/                         I-DATE
202                       I-DATE
##5                       I-DATE
.                         O
contact                   O
:                         O
98                        B-PHONE
##7                       I-PHONE
##65                      I-PHONE
##43                      I-PHONE
##21                      I-PHONE
##0                       I-PHONE
.                         O
email                     O
:             

In [ ]:
def align_example(example):
    text = example["text"]

    encoded = tokenizer(
        text,
        truncation=True,
        return_offsets_mapping=True
    )

    aligned_labels = []

    for token_start, token_end in encoded["offset_mapping"]:

        if token_start == token_end:
            aligned_labels.append(-100)
            continue

        assigned_label = "O"

        for entity in example["annotations"]:
            start = entity["start"]
            end = entity["end"]

            if token_start < end and token_end > start:
                if token_start <= start < token_end:
                    assigned_label = "B-" + entity["label"]
                else:
                    assigned_label = "I-" + entity["label"]
                break

        aligned_labels.append(label2id[assigned_label])

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": aligned_labels
    }


aligned_dataset = [
    align_example(example)
    for example in annotated_data
]

print("Total notes aligned:", len(aligned_dataset))

for i, item in enumerate(aligned_dataset):
    print(
        f"NOTE {i+1}: Tokens = {len(item['input_ids'])}, "
        f"Labels = {len(item['labels'])}"
    )

    assert len(item["input_ids"]) == len(item["labels"])

print("All three notes passed alignment validation!")

Total notes aligned: 3
NOTE 1: Tokens = 44, Labels = 44
NOTE 2: Tokens = 36, Labels = 36
NOTE 3: Tokens = 30, Labels = 30
All three notes passed alignment validation!


In [ ]:
from faker import Faker
import random

fake = Faker()
Faker.seed(42)
random.seed(42)

def generate_clinical_note(note_id):
    name = fake.name()
    doctor = fake.name()
    city = fake.city()
    street = fake.street_address()
    date = fake.date_between(
        start_date="-5y",
        end_date="today"
    ).strftime("%d/%m/%Y")
    phone = fake.numerify("##########")
    email = fake.email()
    age = random.randint(18, 89)
    mrn = "MRN" + fake.numerify("######")

    text = (
        f"Patient {name}, aged {age}, visited "
        f"{city} Medical Center on {date}. "
        f"Contact: {phone}. Email: {email}. "
        f"Medical record number: {mrn}. "
        f"Dr. {doctor} reviewed the patient."
    )

    entities = [
        (name, "NAME"),
        (city, "LOCATION"),
        ("Medical Center", "LOCATION"),
        (date, "DATE"),
        (phone, "PHONE"),
        (email, "EMAIL"),
        (mrn, "MRN"),
        ("Dr. " + doctor, "NAME")
    ]

    return {
        "id": note_id,
        "text": text,
        "entities": entities
    }


synthetic_examples = [
    generate_clinical_note(f"SYNTH_{i:03d}")
    for i in range(1, 101)
]

print("Synthetic notes generated:", len(synthetic_examples))
print("\nSample note:\n")
print(synthetic_examples[0]["text"])

Synthetic notes generated: 100

Sample note:

Patient Allison Hill, aged 32, visited New Roberttown Medical Center on 26/07/2024. Contact: 3794026542. Email: susanrogers@example.org. Medical record number: MRN615594. Dr. Noah Rhodes reviewed the patient.


In [ ]:
# Correct the synthetic entity annotations
for example in synthetic_examples:
    text = example["text"]

    # Extract the city and facility phrase from the generated note
    location = next(
        entity[0] for entity in example["entities"]
        if entity[1] == "LOCATION"
    )

    # Replace separate city/facility labels with one complete location
    example["entities"] = [
        (value, label)
        for value, label in example["entities"]
        if label != "LOCATION"
    ]

    example["entities"].append(
        (location + " Medical Center", "LOCATION")
    )

# Convert character annotations into verified offsets
synthetic_annotated_data = [
    create_annotations(example)
    for example in synthetic_examples
]

# Verify every annotation
for item in synthetic_annotated_data:
    for entity in item["annotations"]:
        extracted = item["text"][entity["start"]:entity["end"]]
        assert extracted == entity["text"]

print("Synthetic annotated notes:", len(synthetic_annotated_data))
print("All synthetic annotation offsets verified!")

print("\nSample annotations:")
for entity in synthetic_annotated_data[0]["annotations"]:
    print(entity["text"], "->", entity["label"])

Synthetic annotated notes: 100
All synthetic annotation offsets verified!

Sample annotations:
Allison Hill -> NAME
26/07/2024 -> DATE
3794026542 -> PHONE
susanrogers@example.org -> EMAIL
MRN615594 -> MRN
Dr. Noah Rhodes -> NAME
New Roberttown Medical Center -> LOCATION


In [ ]:
# Combine the original and synthetic examples
combined_data = annotated_data + synthetic_annotated_data

# Shuffle the examples
random.seed(42)
random.shuffle(combined_data)

# Convert every note into token-level labels
aligned_combined_data = [
    align_example(example)
    for example in combined_data
]

print("Original notes:", len(annotated_data))
print("Synthetic notes:", len(synthetic_annotated_data))
print("Total training examples:", len(aligned_combined_data))

# Validate token-label alignment
for item in aligned_combined_data:
    assert len(item["input_ids"]) == len(item["labels"])

print("All token-label pairs verified successfully!")

Original notes: 3
Synthetic notes: 100
Total training examples: 103
All token-label pairs verified successfully!


In [ ]:
from sklearn.model_selection import train_test_split

train_data, validation_data = train_test_split(
    aligned_combined_data,
    test_size=0.20,
    random_state=42
)

print("Training examples:", len(train_data))
print("Validation examples:", len(validation_data))
print("Total examples:", len(train_data) + len(validation_data))

assert len(train_data) + len(validation_data) == 103

print("Dataset split completed successfully!")

Training examples: 82
Validation examples: 21
Total examples: 103
Dataset split completed successfully!


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class PHIDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        item = self.data[index]

        return {
            "input_ids": torch.tensor(
                item["input_ids"], dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                item["attention_mask"], dtype=torch.long
            ),
            "labels": torch.tensor(
                item["labels"], dtype=torch.long
            )
        }

train_dataset = PHIDataset(train_data)
validation_dataset = PHIDataset(validation_data)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(validation_dataset))
print("First training example:")
print(train_dataset[0])

Training dataset: 82
Validation dataset: 21
First training example:
{'input_ids': tensor([  101,  5776,  2585,  9258,  1010,  4793,  5764,  1010,  4716,  2264,
         9606, 14359,  2966,  2415,  2006,  2538,  1013,  5718,  1013, 16798,
         2549,  1012,  3967,  1024, 29269, 27814, 21486, 16703,  2509,  1012,
        10373,  1024, 29017, 11631, 15551,  1030,  2742,  1012,  5658,  1012,
         2966,  2501,  2193,  1024,  2720,  2078, 28311,  2620, 24594,  2487,
         1012,  2852,  1012, 10972, 28895, 20315,  2015,  8182,  1996,  5776,
         1012,   102]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]), 'labels': tensor([-100,    0,    1,    2,    0,    0,    0,    0,    0,    3,    4,    4,
           4,    4,    0,    5,    6,    6,    6,    6,    6,    0,    0,    0,
           9,   10,   10, 

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=data_collator
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=data_collator
)

# Check one training batch
batch = next(iter(train_loader))

print("Input IDs shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)
print("Labels shape:", batch["labels"].shape)
print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

assert batch["input_ids"].shape == batch["labels"].shape

print("Batch preparation successful!")

Input IDs shape: torch.Size([8, 63])
Attention mask shape: torch.Size([8, 63])
Labels shape: torch.Size([8, 63])
Training batches: 11
Validation batches: 3
Batch preparation successful!


In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(PHI_LABELS),
    id2label=id2label,
    label2id=label2id
)

# Calculate exact model parameter count
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Model:", MODEL_NAME)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("Number of BIO labels:", len(PHI_LABELS))

assert total_parameters <= 1_000_000_000

print("Model loaded successfully!")
print("Assessment parameter limit: PASSED")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: distilbert-base-uncased
Total parameters: 66,383,643
Trainable parameters: 66,383,643
Number of BIO labels: 27
Model loaded successfully!
Assessment parameter limit: PASSED


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./phi_deid_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none"
)

print("Training configuration created!")
print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Batch size:", training_args.per_device_train_batch_size)

Training configuration created!
Epochs: 5
Learning rate: 2e-05
Batch size: 8


In [ ]:
import numpy as np
from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score
)

def compute_metrics(eval_pred):
    predictions, true_labels = eval_pred

    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_targets = []

    for pred_sequence, label_sequence in zip(
        predictions, true_labels
    ):
        predicted_tags = []
        actual_tags = []

        for pred, label in zip(pred_sequence, label_sequence):
            if label == -100:
                continue

            predicted_tags.append(id2label[int(pred)])
            actual_tags.append(id2label[int(label)])

        true_predictions.append(predicted_tags)
        true_targets.append(actual_tags)

    return {
        "precision": precision_score(
            true_targets, true_predictions, zero_division=0
        ),
        "recall": recall_score(
            true_targets, true_predictions, zero_division=0
        ),
        "f1": f1_score(
            true_targets, true_predictions, zero_division=0
        )
    }


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer initialized successfully!")
print("Training notes:", len(train_dataset))
print("Validation notes:", len(validation_dataset))
print("Evaluation metrics: Precision, Recall, F1")

Trainer initialized successfully!
Training notes: 82
Validation notes: 21
Evaluation metrics: Precision, Recall, F1


In [ ]:
training_results = trainer.train()

print("\nTraining completed!")

print("Training loss:", training_results.training_loss)

print("\nValidation results:")
evaluation_results = trainer.evaluate()

for key, value in evaluation_results.items():
    if isinstance(value, (int, float)):
        print(f"{key}: {value:.4f}")

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,2.423965,2.055585,0.000000,0.000000,0.000000
2,1.698997,1.373727,0.254237,0.204082,0.226415
3,1.215431,0.925305,0.658065,0.693878,0.675497
4,0.886922,0.675472,0.945946,0.952381,0.949153
5,0.655358,0.595429,0.979730,0.986395,0.983051


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


... (5 outputs left out)


Training completed!
Training loss: 1.4486611799760298

Validation results:


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Precision,Recall,F1
0.655358,0.595429,5,0.979730,0.986395,0.983051


eval_loss: 0.5954
eval_precision: 0.9797
eval_recall: 0.9864
eval_f1: 0.9831


In [ ]:
test_templates = [
    "The patient {name} was admitted to {city} General Hospital on {date}. Phone: {phone}.",
    "On {date}, Dr. {doctor} consulted with {name} at {city} Medical Centre. MRN: {mrn}.",
    "{name} contacted the clinic from {city}. Email: {email}. Appointment date: {date}.",
    "Medical records show that {name}, aged {age}, visited {city} Clinic on {date}.",
    "Patient {name} reported chest pain for 3 days after surgery at {city} Hospital. Contact: {phone}."
]

test_examples = []

for i in range(20):
    name = fake.name()
    doctor = fake.name()
    city = fake.city()
    date = fake.date_between(
        start_date="-5y", end_date="today"
    ).strftime("%d/%m/%Y")

    phone = fake.numerify("##########")
    email = fake.email()
    mrn = "MRN" + fake.numerify("######")
    age = random.randint(90, 99)

    template = test_templates[i % len(test_templates)]

    text = template.format(
        name=name,
        doctor=doctor,
        city=city,
        date=date,
        phone=phone,
        email=email,
        mrn=mrn,
        age=age
    )

    entities = [(name, "NAME"), (city, "LOCATION")]

    if "{date}" in template:
        entities.append((date, "DATE"))

    if "{doctor}" in template:
        entities.append(("Dr. " + doctor, "NAME"))

    if "{phone}" in template:
        entities.append((phone, "PHONE"))

    if "{email}" in template:
        entities.append((email, "EMAIL"))

    if "{mrn}" in template:
        entities.append((mrn, "MRN"))

    if "{age}" in template:
        entities.append((str(age), "AGE"))

    test_examples.append({
        "id": f"TEST_{i+1:03d}",
        "text": text,
        "entities": entities
    })

test_annotated_data = [
    create_annotations(example)
    for example in test_examples
]

print("Independent test notes:", len(test_annotated_data))

for item in test_annotated_data[:5]:
    print("\n", item["text"])
    print("Annotations:", [
        (e["text"], e["label"])
        for e in item["annotations"]
    ])

print("\nCandidate test annotations generated successfully!")

Independent test notes: 20

 The patient George Pierce was admitted to Port Christopherberg General Hospital on 14/03/2024. Phone: 9611435190.
Annotations: [('George Pierce', 'NAME'), ('Port Christopherberg', 'LOCATION'), ('14/03/2024', 'DATE'), ('9611435190', 'PHONE')]

 On 08/01/2022, Dr. Donna Johnson consulted with Dakota Sandoval at New Gavin Medical Centre. MRN: MRN184895.
Annotations: [('Dakota Sandoval', 'NAME'), ('New Gavin', 'LOCATION'), ('08/01/2022', 'DATE'), ('Dr. Donna Johnson', 'NAME'), ('MRN184895', 'MRN')]

 Robert Perez contacted the clinic from West Gregoryhaven. Email: kchurch@example.com. Appointment date: 17/05/2022.
Annotations: [('Robert Perez', 'NAME'), ('West Gregoryhaven', 'LOCATION'), ('17/05/2022', 'DATE'), ('kchurch@example.com', 'EMAIL')]

 Medical records show that Mr. James Kramer, aged 93, visited Moseshaven Clinic on 09/07/2024.
Annotations: [('Mr. James Kramer', 'NAME'), ('Moseshaven', 'LOCATION'), ('09/07/2024', 'DATE'), ('93', 'AGE')]

 Patient Dal

In [ ]:
facility_suffixes = {
    0: " General Hospital",
    1: " Medical Centre",
    2: "",
    3: " Clinic",
    4: " Hospital"
}

for i, example in enumerate(test_examples):

    suffix = facility_suffixes[i % 5]

    updated_entities = []

    for value, label in example["entities"]:
        if label == "LOCATION":
            value = value + suffix

        updated_entities.append((value, label))

    example["entities"] = updated_entities

# Rebuild annotations
test_annotated_data = [
    create_annotations(example)
    for example in test_examples
]

# Verify offsets
for item in test_annotated_data:
    for entity in item["annotations"]:
        assert (
            item["text"][entity["start"]:entity["end"]]
            == entity["text"]
        )

print("Test notes:", len(test_annotated_data))
print("All annotation offsets verified!")

for item in test_annotated_data:
    print("\n", item["id"], ":", item["text"])
    print([
        (e["text"], e["label"])
        for e in item["annotations"]
    ])

Test notes: 20
All annotation offsets verified!


KeyError: 'id'

In [ ]:
for i, item in enumerate(test_annotated_data):

    print("\n", f"TEST_{i+1:03d}", ":", item["text"])

    print("Annotations:")

    for entity in item["annotations"]:
        print(
            entity["text"],
            "->",
            entity["label"]
        )

print("\nAll 20 test notes displayed successfully!")


 TEST_001 : The patient George Pierce was admitted to Port Christopherberg General Hospital on 14/03/2024. Phone: 9611435190.
Annotations:
George Pierce -> NAME
Port Christopherberg General Hospital -> LOCATION
14/03/2024 -> DATE
9611435190 -> PHONE

 TEST_002 : On 08/01/2022, Dr. Donna Johnson consulted with Dakota Sandoval at New Gavin Medical Centre. MRN: MRN184895.
Annotations:
Dakota Sandoval -> NAME
New Gavin Medical Centre -> LOCATION
08/01/2022 -> DATE
Dr. Donna Johnson -> NAME
MRN184895 -> MRN

 TEST_003 : Robert Perez contacted the clinic from West Gregoryhaven. Email: kchurch@example.com. Appointment date: 17/05/2022.
Annotations:
Robert Perez -> NAME
West Gregoryhaven -> LOCATION
17/05/2022 -> DATE
kchurch@example.com -> EMAIL

 TEST_004 : Medical records show that Mr. James Kramer, aged 93, visited Moseshaven Clinic on 09/07/2024.
Annotations:
Mr. James Kramer -> NAME
Moseshaven Clinic -> LOCATION
09/07/2024 -> DATE
93 -> AGE

 TEST_005 : Patient Dale Wright reported ches

In [ ]:
# Convert test notes into token-level labels
aligned_test_data = [
    align_example(example)
    for example in test_annotated_data
]

# Create independent test dataset
test_dataset = PHIDataset(aligned_test_data)

# Run model predictions
test_results = trainer.predict(test_dataset)

print("\nINDEPENDENT TEST RESULTS")
print("------------------------")

for key, value in test_results.metrics.items():
    if isinstance(value, (int, float)):
        print(f"{key}: {value:.4f}")

print("\nTest notes evaluated:", len(test_dataset))
print("Test evaluation completed!")


INDEPENDENT TEST RESULTS
------------------------
test_loss: 1.0579
test_precision: 0.5610
test_recall: 0.5750
test_f1: 0.5679
test_runtime: 3.4329
test_samples_per_second: 5.8260
test_steps_per_second: 0.8740

Test notes evaluated: 20
Test evaluation completed!


In [ ]:
import numpy as np
from seqeval.metrics import classification_report, get_entities

# Get predictions from independent test results
predictions = np.argmax(test_results.predictions, axis=2)
true_labels = test_results.label_ids

predicted_sequences = []
actual_sequences = []

for pred_sequence, label_sequence in zip(predictions, true_labels):

    pred_tags = []
    true_tags = []

    for pred, label in zip(pred_sequence, label_sequence):

        if label == -100:
            continue

        pred_tags.append(id2label[int(pred)])
        true_tags.append(id2label[int(label)])

    predicted_sequences.append(pred_tags)
    actual_sequences.append(true_tags)

# Per-category evaluation
print("PER-CATEGORY PERFORMANCE")
print(classification_report(
    actual_sequences,
    predicted_sequences,
    digits=4,
    zero_division=0
))

# Calculate missed entities and note-level leak rate
total_entities = 0
missed_entities = 0
leaked_notes = 0

for actual, predicted in zip(
    actual_sequences,
    predicted_sequences
):
    actual_entities = set(get_entities(actual))
    predicted_entities = set(get_entities(predicted))

    missed = actual_entities - predicted_entities

    total_entities += len(actual_entities)
    missed_entities += len(missed)

    if missed:
        leaked_notes += 1

leak_rate = leaked_notes / len(actual_sequences)

print("\nLEAK ANALYSIS")
print("Total actual entities:", total_entities)
print("Missed entities:", missed_entities)
print("Notes with at least one missed entity:", leaked_notes)
print("Total test notes:", len(actual_sequences))
print(f"Leak rate: {leak_rate:.2%}")

ImportError: cannot import name 'get_entities' from 'seqeval.metrics' (/usr/local/lib/python3.13/dist-packages/seqeval/metrics/__init__.py)

In [ ]:
import numpy as np
from seqeval.metrics import classification_report

def extract_entities(tags):
    entities = set()
    current_type = None
    start = None

    for i, tag in enumerate(tags + ["O"]):

        if tag == "O":
            if current_type is not None:
                entities.add((start, i, current_type))
                current_type = None
            continue

        prefix, entity_type = tag.split("-", 1)

        if prefix == "B" or entity_type != current_type:
            if current_type is not None:
                entities.add((start, i, current_type))

            current_type = entity_type
            start = i

    return entities


predictions = np.argmax(test_results.predictions, axis=2)
true_labels = test_results.label_ids

predicted_sequences = []
actual_sequences = []

for pred_sequence, label_sequence in zip(predictions, true_labels):

    pred_tags = []
    true_tags = []

    for pred, label in zip(pred_sequence, label_sequence):

        if label == -100:
            continue

        pred_tags.append(id2label[int(pred)])
        true_tags.append(id2label[int(label)])

    predicted_sequences.append(pred_tags)
    actual_sequences.append(true_tags)


print("PER-CATEGORY PERFORMANCE")
print(classification_report(
    actual_sequences,
    predicted_sequences,
    digits=4,
    zero_division=0
))


total_entities = 0
missed_entities = 0
leaked_notes = 0

for actual, predicted in zip(
    actual_sequences,
    predicted_sequences
):

    actual_entities = extract_entities(actual)
    predicted_entities = extract_entities(predicted)

    missed = actual_entities - predicted_entities

    total_entities += len(actual_entities)
    missed_entities += len(missed)

    if missed:
        leaked_notes += 1


leak_rate = leaked_notes / len(actual_sequences)

print("\nLEAK ANALYSIS")
print("Total actual entities:", total_entities)
print("Missed entities:", missed_entities)
print("Notes with missed entities:", leaked_notes)
print("Total test notes:", len(actual_sequences))
print(f"Leak rate: {leak_rate:.2%}")

PER-CATEGORY PERFORMANCE
              precision    recall  f1-score   support

         AGE     0.0000    0.0000    0.0000         4
        DATE     1.0000    1.0000    1.0000        16
       EMAIL     0.5000    1.0000    0.6667         4
    LOCATION     0.4667    0.3500    0.4000        20
         MRN     0.1667    0.2500    0.2000         4
        NAME     0.4615    0.5000    0.4800        24
       PHONE     0.5455    0.7500    0.6316         8

   micro avg     0.5610    0.5750    0.5679        80
   macro avg     0.4486    0.5500    0.4826        80
weighted avg     0.5430    0.5750    0.5505        80


LEAK ANALYSIS
Total actual entities: 80
Missed entities: 34
Notes with missed entities: 17
Total test notes: 20
Leak rate: 85.00%


In [ ]:
def get_entity_spans(tags, offsets):
    spans = []
    current_type = None
    start = None

    for i, tag in enumerate(tags + ["O"]):

        if tag == "O":
            if current_type is not None:
                spans.append((
                    offsets[start][0],
                    offsets[i-1][1],
                    current_type
                ))
                current_type = None
            continue

        prefix, entity_type = tag.split("-", 1)

        if prefix == "B" or entity_type != current_type:
            if current_type is not None:
                spans.append((
                    offsets[start][0],
                    offsets[i-1][1],
                    current_type
                ))

            current_type = entity_type
            start = i

    return spans


print("TEST ERROR INSPECTION")
print("=" * 60)

for i, example in enumerate(test_annotated_data):

    encoded = tokenizer(
        example["text"],
        return_offsets_mapping=True,
        truncation=True
    )

    offsets = [
        offset for offset in encoded["offset_mapping"]
        if offset[0] != offset[1]
    ]

    gold_spans = {
        (e["start"], e["end"], e["label"])
        for e in example["annotations"]
    }

    predicted_spans = set(
        get_entity_spans(
            predicted_sequences[i],
            offsets
        )
    )

    missed = gold_spans - predicted_spans
    extra = predicted_spans - gold_spans

    if missed or extra:
        print(f"\nTEST_{i+1:03d}")

        for start, end, label in sorted(missed):
            print(
                "MISSED:",
                example["text"][start:end],
                "->",
                label
            )

        for start, end, label in sorted(extra):
            print(
                "EXTRA:",
                example["text"][start:end],
                "->",
                label
            )

print("\nError inspection completed!")

TEST ERROR INSPECTION

TEST_001
MISSED: George Pierce -> NAME
MISSED: Port Christopherberg General Hospital -> LOCATION
EXTRA: Pierce -> NAME
EXTRA: Christopherberg General Hospital -> LOCATION

TEST_002
MISSED: Dr. Donna Johnson -> NAME
MISSED: Dakota Sandoval -> NAME
MISSED: New Gavin Medical Centre -> LOCATION
MISSED: MRN184895 -> MRN
EXTRA: . Donna Johnson -> NAME
EXTRA: Gavin Medical Centre -> LOCATION
EXTRA: MRN18489 -> MRN
EXTRA: 5 -> PHONE

TEST_003
MISSED: West Gregoryhaven -> LOCATION
EXTRA: Gregory -> EMAIL

TEST_004
MISSED: Mr. James Kramer -> NAME
MISSED: 93 -> AGE
EXTRA: James Kramer -> NAME

TEST_005
MISSED: Lake Stephaniefort Hospital -> LOCATION
EXTRA: Stephaniefort Hospital -> LOCATION

TEST_007
MISSED: Dr. James Hayes -> NAME
MISSED: New Gina Medical Centre -> LOCATION
MISSED: MRN885744 -> MRN
EXTRA: . James Hayes -> NAME
EXTRA: Gina Medical Centre -> LOCATION
EXTRA: MRN885 -> MRN
EXTRA: 744 -> PHONE

TEST_008
MISSED: Combsville -> LOCATION
EXTRA: Combsville -> EMAIL

In [ ]:
def generate_improved_note(note_id):

    patient = fake.name()
    doctor = fake.name()
    city = fake.city()

    facility = city + " " + random.choice(facility_types)

    date = fake.date_between(
        start_date="-5y",
        end_date="today"
    ).strftime("%d/%m/%Y")

    phone = random.choice(phone_formats)()
    mrn = random.choice(mrn_formats)()
    email = fake.email()

    age = random.choice(
        list(range(90, 101))
        if random.random() < 0.4
        else list(range(18, 90))
    )

    age_text = str(age)

    templates = [
        f"Patient {patient}, aged {age_text}, was admitted to {facility} on {date}. Contact: {phone}. Record: {mrn}.",

        f"On {date}, Dr. {doctor} examined {patient} at {facility}. Phone: {phone}. Email: {email}.",

        f"{patient}, a {age_text}-year-old patient, visited {facility} for consultation on {date}.",

        f"Clinical report for {patient}. Facility: {facility}. Appointment: {date}. Medical record number: {mrn}.",

        f"Dr. {doctor} reviewed the case of {patient} at {facility}. The visit occurred on {date}. Contact number: {phone}.",

        f"Please contact patient {patient} at {phone}. The patient visited {facility} on {date}. Reference ID: {mrn}.",

        f"Patient details: {patient}; Age: {age_text}; Hospital: {facility}; Visit date: {date}; Email: {email}.",

        f"The medical team at {facility} evaluated {patient} on {date}. Dr. {doctor} documented the consultation. MRN: {mrn}."
    ]

    text = random.choice(templates)

    entities = [
        (patient, "NAME"),
        (facility, "LOCATION"),
        (date, "DATE")
    ]

    if "Dr. " + doctor in text:
        entities.append(("Dr. " + doctor, "NAME"))

    if phone in text:
        entities.append((phone, "PHONE"))

    if email in text:
        entities.append((email, "EMAIL"))

    if mrn in text:
        entities.append((mrn, "MRN"))

    if age > 89 and age_text in text:
        entities.append((age_text, "AGE"))

    return {
        "id": note_id,
        "text": text,
        "entities": entities
    }


improved_examples = [
    generate_improved_note(f"IMPROVED_{i:03d}")
    for i in range(1, 201)
]

improved_annotated_data = [
    create_annotations(example)
    for example in improved_examples
]

print("New training notes:", len(improved_annotated_data))
print("Sample note:", improved_annotated_data[0]["text"])
print("\nSample annotations:")
print(improved_annotated_data[0]["annotations"])

print(
    "\nAge examples labelled:",
    sum(
        1 for example in improved_annotated_data
        for entity in example["annotations"]
        if entity["label"] == "AGE"
    )
)

New training notes: 200
Sample note: Alexis Miller, a 92-year-old patient, visited Port Sandrashire General Hospital for consultation on 17/04/2025.

Sample annotations:
[{'start': 0, 'end': 13, 'label': 'NAME', 'text': 'Alexis Miller'}, {'start': 46, 'end': 79, 'label': 'LOCATION', 'text': 'Port Sandrashire General Hospital'}, {'start': 100, 'end': 110, 'label': 'DATE', 'text': '17/04/2025'}, {'start': 17, 'end': 19, 'label': 'AGE', 'text': '92'}]

Age examples labelled: 27


In [ ]:
from collections import Counter

label_counts = Counter()
annotation_errors = []

for i, example in enumerate(improved_annotated_data):

    text = example["text"]

    for entity in example["annotations"]:
        start = entity["start"]
        end = entity["end"]
        label = entity["label"]
        value = entity["text"]

        # Verify character offsets
        if text[start:end] != value:
            annotation_errors.append({
                "note": i + 1,
                "entity": value,
                "label": label,
                "actual_text": text[start:end]
            })

        label_counts[label] += 1

print("ANNOTATION VALIDATION")
print("=" * 40)
print("Total notes:", len(improved_annotated_data))
print("Total entities:", sum(label_counts.values()))
print("Annotation errors:", len(annotation_errors))

print("\nENTITY COUNTS")
for label, count in sorted(label_counts.items()):
    print(f"{label}: {count}")

print("\nAGE LABEL CHECK")
print("AGE annotations:", label_counts["AGE"])

if annotation_errors:
    print("\nFirst errors:")
    print(annotation_errors[:5])
else:
    print("\nAll annotation offsets verified successfully!")
    # Combine original and improved training data
combined_data = annotated_data + synthetic_annotated_data + improved_annotated_data

print("Total training notes:", len(combined_data))

# Shuffle the combined data
random.seed(42)
random.shuffle(combined_data)

# Convert character annotations into token labels
aligned_combined_data = [
    align_example(example)
    for example in combined_data
]

print("Aligned notes:", len(aligned_combined_data))

# Verify token and label lengths
for item in aligned_combined_data:
    assert len(item["input_ids"]) == len(item["labels"])
    assert len(item["input_ids"]) == len(item["attention_mask"])

print("All token-label alignments verified successfully!")
from sklearn.model_selection import train_test_split

train_data, validation_data = train_test_split(
    aligned_combined_data,
    test_size=0.20,
    random_state=42
)

train_dataset = PHIDataset(train_data)
validation_dataset = PHIDataset(validation_data)

print("TRAINING DATA:", len(train_dataset))
print("VALIDATION DATA:", len(validation_dataset))
print("INDEPENDENT TEST DATA:", len(test_dataset))
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=data_collator
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=data_collator
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

batch = next(iter(train_loader))

print("\nBatch input shape:", batch["input_ids"].shape)
print("Batch label shape:", batch["labels"].shape)
from transformers import AutoModelForTokenClassification

MODEL_NAME = "distilbert-base-uncased"

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(PHI_LABELS),
    id2label=id2label,
    label2id=label2id
)

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Fresh model loaded successfully!")
print("Model:", MODEL_NAME)
print("Number of labels:", len(PHI_LABELS))
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./phi_deid_improved",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Improved training configuration ready!")
print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Batch size:", training_args.per_device_train_batch_size)
training_results = trainer.train()

print("\nIMPROVED TRAINING COMPLETED")
print("=" * 40)
print("Training loss:", training_results.training_loss)
# Evaluate the improved model on the untouched test set
improved_test_results = trainer.predict(test_dataset)

print("\nIMPROVED MODEL — TEST RESULTS")
print("=" * 45)

print("Test Loss:", improved_test_results.metrics["test_loss"])
print("Test Precision:", improved_test_results.metrics["test_precision"])
print("Test Recall:", improved_test_results.metrics["test_recall"])
print("Test F1:", improved_test_results.metrics["test_f1"])

ANNOTATION VALIDATION
Total notes: 200
Total entities: 953
Annotation errors: 0

ENTITY COUNTS
AGE: 27
DATE: 200
EMAIL: 46
LOCATION: 200
MRN: 103
NAME: 273
PHONE: 104

AGE LABEL CHECK
AGE annotations: 27

All annotation offsets verified successfully!
Total training notes: 303
Aligned notes: 303
All token-label alignments verified successfully!
TRAINING DATA: 242
VALIDATION DATA: 61
INDEPENDENT TEST DATA: 20
Training batches: 31
Validation batches: 8

Batch input shape: torch.Size([8, 65])
Batch label shape: torch.Size([8, 65])


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fresh model loaded successfully!
Model: distilbert-base-uncased
Number of labels: 27
Total parameters: 66,383,643
Trainable parameters: 66,383,643
Improved training configuration ready!
Epochs: 5
Learning rate: 2e-05
Batch size: 8


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,1.312516,0.903321,0.492492,0.498480,0.495468
2,0.290742,0.164357,0.975460,0.966565,0.970992
3,0.087115,0.057138,0.990769,0.978723,0.984709
4,0.055490,0.040956,0.990798,0.981763,0.986260
5,0.047423,0.037212,0.993846,0.981763,0.987768


... (8 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


IMPROVED TRAINING COMPLETED
Training loss: 0.5344075808601995


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



IMPROVED MODEL — TEST RESULTS
Test Loss: 0.07569979131221771
Test Precision: 0.961038961038961
Test Recall: 0.925
Test F1: 0.9426751592356688


In [ ]:
import numpy as np
from seqeval.metrics import classification_report

# Get predictions from improved model
predictions = np.argmax(
    improved_test_results.predictions,
    axis=2
)

true_labels = improved_test_results.label_ids

predicted_sequences = []
actual_sequences = []

for pred_sequence, label_sequence in zip(predictions, true_labels):

    pred_tags = []
    true_tags = []

    for pred, label in zip(pred_sequence, label_sequence):

        if label == -100:
            continue

        pred_tags.append(id2label[int(pred)])
        true_tags.append(id2label[int(label)])

    predicted_sequences.append(pred_tags)
    actual_sequences.append(true_tags)


# Per-category performance
print("IMPROVED MODEL - PER-CATEGORY PERFORMANCE")
print("=" * 55)

print(classification_report(
    actual_sequences,
    predicted_sequences,
    digits=4,
    zero_division=0
))


# Note-level leak analysis
total_entities = 0
missed_entities = 0
leaked_notes = 0

for actual, predicted in zip(
    actual_sequences,
    predicted_sequences
):

    actual_entities = extract_entities(actual)
    predicted_entities = extract_entities(predicted)

    missed = actual_entities - predicted_entities

    total_entities += len(actual_entities)
    missed_entities += len(missed)

    if missed:
        leaked_notes += 1

leak_rate = leaked_notes / len(actual_sequences) * 100

print("\nIMPROVED MODEL - LEAK ANALYSIS")
print("=" * 45)

print("Total actual entities:", total_entities)
print("Missed entities:", missed_entities)
print("Notes with missed entities:", leaked_notes)
print("Total test notes:", len(actual_sequences))
print(f"Leak rate: {leak_rate:.2f}%")
print("AGE DETECTION ERROR INSPECTION")
print("=" * 60)

for i, example in enumerate(test_annotated_data):

    age_entities = [
        entity for entity in example["annotations"]
        if entity["label"] == "AGE"
    ]

    if not age_entities:
        continue

    encoded = tokenizer(
        example["text"],
        return_offsets_mapping=True,
        truncation=True
    )

    offsets = [
        offset for offset in encoded["offset_mapping"]
        if offset[0] != offset[1]
    ]

    predicted_tags = predicted_sequences[i]

    print(f"\nTEST NOTE {i+1:03d}")
    print("Text:", example["text"])

    for entity in age_entities:

        start = entity["start"]
        end = entity["end"]

        print("\nActual AGE:", entity["text"])

        for token_index, (token_start, token_end) in enumerate(offsets):

            if token_start < end and token_end > start:

                token_text = example["text"][token_start:token_end]

                print(
                    f"Token: {token_text:12s}",
                    f"Predicted label: {predicted_tags[token_index]}"
                )

print("\nAge inspection completed!")
from collections import Counter

def count_age_labels(dataset_data):

    counts = Counter()

    for example in dataset_data:
        for label_id in example["labels"]:
            if label_id == -100:
                continue

            label = id2label[int(label_id)]

            if "AGE" in label:
                counts[label] += 1

    return counts


train_age_counts = count_age_labels(train_data)
val_age_counts = count_age_labels(validation_data)

print("AGE LABEL DISTRIBUTION")
print("=" * 40)

print("Training B-AGE:", train_age_counts["B-AGE"])
print("Training I-AGE:", train_age_counts["I-AGE"])

print("\nValidation B-AGE:", val_age_counts["B-AGE"])
print("Validation I-AGE:", val_age_counts["I-AGE"])
import random

random.seed(200)
Faker.seed(200)

age_templates = [
    "Patient {name}, aged {age}, was admitted to {facility}.",
    "{name} is a {age}-year-old patient receiving treatment at {facility}.",
    "The patient is {age} years old and visited {facility}.",
    "Clinical assessment: {name}, age {age}.",
    "Age of patient {name}: {age}. Hospital: {facility}.",
    "{name}, an elderly patient aged {age}, was examined at {facility}.",
    "Patient information: Name {name}; Age {age}; Facility {facility}.",
    "The medical team reviewed {name}, who is {age} years of age.",
    "An individual aged {age} years was evaluated at {facility}.",
    "Patient {name}, {age} years old, presented for consultation."
]

focused_age_examples = []

for i in range(100):

    name = fake.name()
    age = random.randint(90, 105)
    facility = fake.city() + " " + random.choice(facility_types)

    template = random.choice(age_templates)

    text = template.format(
        name=name,
        age=age,
        facility=facility
    )

    entities = [
        (name, "NAME"),
        (str(age), "AGE")
    ]

    if facility in text:
        entities.append((facility, "LOCATION"))

    focused_age_examples.append({
        "id": f"AGE_FOCUS_{i+1:03d}",
        "text": text,
        "entities": entities
    })

focused_age_annotated = [
    create_annotations(example)
    for example in focused_age_examples
]

print("Focused age examples:", len(focused_age_annotated))
print("\nSample:")
print(focused_age_annotated[0]["text"])
print("\nAnnotations:")
print(focused_age_annotated[0]["annotations"])
focused_age_examples = []

import random

random.seed(200)
Faker.seed(200)

age_templates = [
    "Patient {name}, aged {age}, was admitted to {facility}.",
    "{name} is a {age}-year-old patient receiving treatment at {facility}.",
    "The patient is {age} years old and visited {facility}.",
    "Clinical assessment: {name}, age {age}.",
    "Age of patient {name}: {age}. Hospital: {facility}.",
    "{name}, an elderly patient aged {age}, was examined at {facility}.",
    "Patient information: Name {name}; Age {age}; Facility {facility}.",
    "The medical team reviewed {name}, who is {age} years of age.",
    "An individual aged {age} years was evaluated at {facility}.",
    "Patient {name}, {age} years old, presented for consultation."
]

focused_age_examples = []

for i in range(100):
    name = fake.name()
    age = random.randint(90, 105)
    facility = fake.city() + " " + random.choice(facility_types)

    text = random.choice(age_templates).format(
        name=name,
        age=age,
        facility=facility
    )

    entities = [(str(age), "AGE")]

    for value, label in [
        (name, "NAME"),
        (facility, "LOCATION")
    ]:
        if value in text:
            entities.append((value, label))

    focused_age_examples.append({
        "id": f"AGE_FOCUS_{i+1:03d}",
        "text": text,
        "entities": entities
    })

focused_age_annotated = [
    create_annotations(example)
    for example in focused_age_examples
]

print("Focused age examples:", len(focused_age_annotated))
print("Total AGE annotations:", sum(
    1 for example in focused_age_annotated
    for annotation in example["annotations"]
    if annotation["label"] == "AGE"
))

print("\nSample note:")
print(focused_age_annotated[0]["text"])
print("\nAnnotations:")
print(focused_age_annotated[0]["annotations"])import random

random.seed(200)
Faker.seed(200)

age_templates = [
    "Patient {name}, aged {age}, was admitted to {facility}.",
    "{name} is a {age}-year-old patient receiving treatment at {facility}.",
    "The patient is {age} years old and visited {facility}.",
    "Clinical assessment: {name}, age {age}.",
    "Age of patient {name}: {age}. Hospital: {facility}.",
    "{name}, an elderly patient aged {age}, was examined at {facility}.",
    "Patient information: Name {name}; Age {age}; Facility {facility}.",
    "The medical team reviewed {name}, who is {age} years of age.",
    "An individual aged {age} years was evaluated at {facility}.",
    "Patient {name}, {age} years old, presented for consultation."
]

focused_age_examples = []

for i in range(100):
    name = fake.name()
    age = random.randint(90, 105)
    facility = fake.city() + " " + random.choice(facility_types)

    text = random.choice(age_templates).format(
        name=name,
        age=age,
        facility=facility
    )

    entities = [(str(age), "AGE")]

    for value, label in [
        (name, "NAME"),
        (facility, "LOCATION")
    ]:
        if value in text:
            entities.append((value, label))

    focused_age_examples.append({
        "id": f"AGE_FOCUS_{i+1:03d}",
        "text": text,
        "entities": entities
    })

focused_age_annotated = [
    create_annotations(example)
    for example in focused_age_examples
]

print("Focused age examples:", len(focused_age_annotated))
print("Total AGE annotations:", sum(
    1 for example in focused_age_annotated
    for annotation in example["annotations"]
    if annotation["label"] == "AGE"
))

print("\nSample note:")
print(focused_age_annotated[0]["text"])
print("\nAnnotations:")
print(focused_age_annotated[0]["annotations"])

SyntaxError: invalid syntax (2513250822.py, line 274)

In [ ]:
print(focused_age_annotated[0]["annotations"])import random

SyntaxError: invalid syntax (3949899054.py, line 1)

In [ ]:
!pip install Faker seqeval transformers datasets accelerate scikit-learn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.4 MB/s  0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16250 sha256=cddded138d1c16608d049731e1eed8fa6bcd12fe5cafc2bed30e3e8ad7e44cc6
  Stored in directory: /root/.cache/pip/wheels/14/cf/a7/8f28ef376d707ff10e3922899482a2f23ef3002f8a952f47ac
Successfully built seqeval
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [seqeval]


In [ ]:
from faker import Faker
import random
import numpy as np

fake = Faker()
Faker.seed(200)
random.seed(200)

print("All imports successful!")
print("Test name:", fake.name())

All imports successful!
Test name: Dalton Fletcher


In [ ]:
from faker import Faker
import random
import numpy as np

fake = Faker()
Faker.seed(200)
random.seed(200)

print("All imports successful!")
print("Test name:", fake.name())

All imports successful!
Test name: Dalton Fletcher


In [ ]:
def create_annotations(example):
    text = example["text"]
    annotations = []

    for value, label in example["entities"]:
        start = text.find(value)

        if start == -1:
            raise ValueError(f"Entity not found: {value}")

        end = start + len(value)

        annotations.append({
            "start": start,
            "end": end,
            "label": label,
            "text": value
        })

    return {
        "id": example["id"],
        "text": text,
        "annotations": annotations
    }

print("create_annotations() restored successfully!")

create_annotations() restored successfully!


In [ ]:
facility_types = [
    "General Hospital",
    "Medical Center",
    "Medical Centre",
    "Clinic",
    "Hospital"
]

print("Facility types restored!")
print(facility_types)

Facility types restored!
['General Hospital', 'Medical Center', 'Medical Centre', 'Clinic', 'Hospital']


In [ ]:
import random

random.seed(200)
Faker.seed(200)

age_templates = [
    "Patient {name}, aged {age}, was admitted to {facility}.",
    "{name} is a {age}-year-old patient receiving treatment at {facility}.",
    "The patient is {age} years old and visited {facility}.",
    "Clinical assessment: {name}, age {age}.",
    "Age of patient {name}: {age}. Hospital: {facility}.",
    "{name}, an elderly patient aged {age}, was examined at {facility}.",
    "Patient information: Name {name}; Age {age}; Facility {facility}.",
    "The medical team reviewed {name}, who is {age} years of age.",
    "An individual aged {age} years was evaluated at {facility}.",
    "Patient {name}, {age} years old, presented for consultation."
]

focused_age_examples = []

for i in range(100):
    name = fake.name()
    age = random.randint(90, 105)
    facility = fake.city() + " " + random.choice(facility_types)

    text = random.choice(age_templates).format(
        name=name,
        age=age,
        facility=facility
    )

    entities = [(str(age), "AGE")]

    if name in text:
        entities.append((name, "NAME"))

    if facility in text:
        entities.append((facility, "LOCATION"))

    focused_age_examples.append({
        "id": f"AGE_FOCUS_{i+1:03d}",
        "text": text,
        "entities": entities
    })

print("Generated examples:", len(focused_age_examples))
print("\nSample:")
print(focused_age_examples[0])

Generated examples: 100

Sample:
{'id': 'AGE_FOCUS_001', 'text': 'Patient Dalton Fletcher, aged 91, was admitted to New Michelle Medical Center.', 'entities': [('91', 'AGE'), ('Dalton Fletcher', 'NAME'), ('New Michelle Medical Center', 'LOCATION')]}


In [ ]:
focused_age_annotated = [
    create_annotations(example)
    for example in focused_age_examples
]

print("Focused age examples:", len(focused_age_annotated))

age_count = sum(
    1
    for example in focused_age_annotated
    for annotation in example["annotations"]
    if annotation["label"] == "AGE"
)

print("Total AGE annotations:", age_count)

print("\nSample note:")
print(focused_age_annotated[0]["text"])

print("\nSample annotations:")
print(focused_age_annotated[0]["annotations"])combined_data = (
    annotated_data
    + synthetic_annotated_data
    + improved_annotated_data
    + focused_age_annotated
)

print("Total training examples:", len(combined_data))
print("Original test examples:", len(test_annotated_data))

SyntaxError: invalid syntax (3187585197.py, line 21)

In [ ]:
combined_data = (
    annotated_data
    + synthetic_annotated_data
    + improved_annotated_data
    + focused_age_annotated
)

print("Total training examples:", len(combined_data))
print("Original test examples:", len(test_annotated_data))


NameError: name 'annotated_data' is not defined

In [ ]:
training_examples = [
    {
        "id": "NOTE_001",
        "text": "John Miller, aged 45, visited Salem City Hospital on 12/03/2025. Phone: 9876543210. Email: john.miller@example.com. Diagnosed with diabetes.",
        "entities": [
            ("John Miller", "NAME"),
            ("Salem City Hospital", "LOCATION"),
            ("12/03/2025", "DATE"),
            ("9876543210", "PHONE"),
            ("john.miller@example.com", "EMAIL")
        ]
    },
    {
        "id": "NOTE_002",
        "text": "Dr. Robert Smith examined Mary Johnson on 15/04/2025. MRN: 123456. Patient reported chest pain for 3 days.",
        "entities": [
            ("Robert Smith", "NAME"),
            ("Mary Johnson", "NAME"),
            ("15/04/2025", "DATE"),
            ("123456", "MRN")
        ]
    },
    {
        "id": "NOTE_003",
        "text": "David Brown, aged 92, visited 25 Lake Road, Salem, Tamil Nadu 636001. Diagnosed with hypertension.",
        "entities": [
            ("David Brown", "NAME"),
            ("25 Lake Road", "LOCATION"),
            ("Salem", "LOCATION"),
            ("636001", "LOCATION"),
            ("92", "AGE")
        ]
    }
]

annotated_data = [
    create_annotations(example)
    for example in training_examples
]

print("Original annotated notes:", len(annotated_data))
print("Original training data restored!")

Original annotated notes: 3
Original training data restored!


In [ ]:
from faker import Faker
import random

fake = Faker()
random.seed(42)
Faker.seed(42)

synthetic_examples = []

for i in range(100):
    name = fake.name()
    city = fake.city()
    date = fake.date(pattern="%d/%m/%Y")
    phone = fake.numerify(text="##########")
    email = fake.email()

    facility = city + " Medical Center"

    text = (
        f"Patient {name} visited {facility} "
        f"on {date}. Phone: {phone}. Email: {email}."
    )

    entities = [
        (name, "NAME"),
        (facility, "LOCATION"),
        (date, "DATE"),
        (phone, "PHONE"),
        (email, "EMAIL")
    ]

    synthetic_examples.append({
        "id": f"SYNTHETIC_{i+1:03d}",
        "text": text,
        "entities": entities
    })

synthetic_annotated_data = [
    create_annotations(example)
    for example in synthetic_examples
]

print("Synthetic examples:", len(synthetic_annotated_data))
print("Sample:")
print(synthetic_annotated_data[0])

Synthetic examples: 100
Sample:
{'id': 'SYNTHETIC_001', 'text': 'Patient Allison Hill visited East Jill Medical Center on 22/06/2008. Phone: 8196001338. Email: hoffmanjennifer@example.net.', 'annotations': [{'start': 8, 'end': 20, 'label': 'NAME', 'text': 'Allison Hill'}, {'start': 29, 'end': 53, 'label': 'LOCATION', 'text': 'East Jill Medical Center'}, {'start': 57, 'end': 67, 'label': 'DATE', 'text': '22/06/2008'}, {'start': 76, 'end': 86, 'label': 'PHONE', 'text': '8196001338'}, {'start': 95, 'end': 122, 'label': 'EMAIL', 'text': 'hoffmanjennifer@example.net'}]}


In [ ]:
import random

random.seed(100)
Faker.seed(100)

def generate_improved_note(note_id):
    name = fake.name()
    doctor = fake.name()
    city = fake.city()
    facility = city + " Medical Center"
    date = fake.date(pattern="%d/%m/%Y")
    phone = fake.numerify(text="##########")
    email = fake.email()
    mrn = str(random.randint(100000, 999999))
    age = random.randint(18, 105)

    templates = [
        "Patient {name}, aged {age}, visited {facility} on {date}. Phone: {phone}.",
        "Dr. {doctor} examined {name} at {facility} on {date}. MRN: {mrn}.",
        "{name} contacted {facility}. Email: {email}. Appointment date: {date}.",
        "Medical records show that {name}, aged {age}, visited {facility} on {date}.",
        "Patient {name} reported chest pain for 3 days after surgery at {facility}. Contact: {phone}.",
        "On {date}, Dr. {doctor} consulted with {name} at {facility}. MRN: {mrn}.",
        "{name}, age {age}, was admitted to {facility}. Contact: {phone}.",
        "Clinical review of {name} at {facility} on {date}. Email: {email}."
    ]

    text = random.choice(templates).format(
        name=name,
        doctor=doctor,
        age=age,
        facility=facility,
        date=date,
        phone=phone,
        email=email,
        mrn=mrn
    )

    entities = []

    for value, label in [
        (name, "NAME"),
        (doctor, "NAME"),
        (facility, "LOCATION"),
        (date, "DATE"),
        (phone, "PHONE"),
        (email, "EMAIL"),
        (mrn, "MRN")
    ]:
        if value in text:
            entities.append((value, label))

    if age > 89 and str(age) in text:
        entities.append((str(age), "AGE"))

    return {
        "id": note_id,
        "text": text,
        "entities": entities
    }


improved_examples = [
    generate_improved_note(f"IMPROVED_{i:03d}")
    for i in range(1, 201)
]

improved_annotated_data = [
    create_annotations(example)
    for example in improved_examples
]

print("Improved examples:", len(improved_annotated_data))
print("Sample note:")
print(improved_annotated_data[0]["text"])
print("\nSample annotations:")
print(improved_annotated_data[0]["annotations"])

Improved examples: 200
Sample note:
Clinical review of Jesse Townsend at New Davidmouth Medical Center on 23/12/1984. Email: nrichmond@example.com.

Sample annotations:
[{'start': 19, 'end': 33, 'label': 'NAME', 'text': 'Jesse Townsend'}, {'start': 37, 'end': 66, 'label': 'LOCATION', 'text': 'New Davidmouth Medical Center'}, {'start': 70, 'end': 80, 'label': 'DATE', 'text': '23/12/1984'}, {'start': 89, 'end': 110, 'label': 'EMAIL', 'text': 'nrichmond@example.com'}]


In [ ]:
combined_data = (
    annotated_data
    + synthetic_annotated_data
    + improved_annotated_data
    + focused_age_annotated
)

print("Total combined examples:", len(combined_data))
print("Focused AGE examples:", len(focused_age_annotated))

print("\nTest dataset available:",
      "test_annotated_data" in globals())

Total combined examples: 403
Focused AGE examples: 100

Test dataset available: False


In [ ]:
import random

random.seed(500)
Faker.seed(500)

test_examples = []

test_templates = [
    "The patient {name} was admitted to {city} General Hospital on {date}. Phone: {phone}.",
    "On {date}, Dr. {doctor} consulted with {name} at {city} Medical Centre. MRN: {mrn}.",
    "{name} contacted the clinic from {city}. Email: {email}. Appointment date: {date}.",
    "Medical records show that {name}, aged {age}, visited {city} Clinic on {date}.",
    "Patient {name} reported chest pain for 3 days after surgery at {city} Hospital. Contact: {phone}."
]

for i in range(20):
    name = fake.name()
    doctor = fake.name()
    city = fake.city()
    date = fake.date(pattern="%d/%m/%Y")
    phone = fake.numerify(text="##########")
    email = fake.email()
    mrn = str(random.randint(100000, 999999))
    age = random.randint(91, 95)

    template = test_templates[i % len(test_templates)]

    text = template.format(
        name=name,
        doctor=doctor,
        city=city,
        date=date,
        phone=phone,
        email=email,
        mrn=mrn,
        age=age
    )

    entities = []

    for value, label in [
        (name, "NAME"),
        (doctor, "NAME"),
        (city, "LOCATION"),
        (date, "DATE"),
        (phone, "PHONE"),
        (email, "EMAIL"),
        (mrn, "MRN")
    ]:
        if value in text:
            entities.append((value, label))

    if "aged" in text:
        entities.append((str(age), "AGE"))

    test_examples.append({
        "id": f"TEST_{i+1:03d}",
        "text": text,
        "entities": entities
    })

test_annotated_data = [
    create_annotations(example)
    for example in test_examples
]

print("Independent test notes:", len(test_annotated_data))
print("\nSample test note:")
print(test_annotated_data[0]["text"])
print("\nSample annotations:")
print(test_annotated_data[0]["annotations"])

Independent test notes: 20

Sample test note:
The patient Tiffany Lane was admitted to West Laurenland General Hospital on 11/01/1971. Phone: 5364015794.

Sample annotations:
[{'start': 12, 'end': 24, 'label': 'NAME', 'text': 'Tiffany Lane'}, {'start': 41, 'end': 56, 'label': 'LOCATION', 'text': 'West Laurenland'}, {'start': 77, 'end': 87, 'label': 'DATE', 'text': '11/01/1971'}, {'start': 96, 'end': 106, 'label': 'PHONE', 'text': '5364015794'}]


In [ ]:
for example in test_examples:
    text = example["text"]

    city = next(
        value for value, label in example["entities"]
        if label == "LOCATION"
    )

    if "General Hospital" in text:
        location = city + " General Hospital"
    elif "Medical Centre" in text:
        location = city + " Medical Centre"
    elif "Clinic on" in text:
        location = city + " Clinic"
    elif "Hospital. Contact:" in text:
        location = city + " Hospital"
    else:
        location = city

    example["entities"] = [
        (value, label)
        for value, label in example["entities"]
        if label != "LOCATION"
    ]
    example["entities"].append((location, "LOCATION"))

test_annotated_data = [
    create_annotations(example)
    for example in test_examples
]

print("Corrected test notes:", len(test_annotated_data))
print("First location annotation:")
print([
    entity for entity in test_annotated_data[0]["annotations"]
    if entity["label"] == "LOCATION"
])

Corrected test notes: 20
First location annotation:
[{'start': 41, 'end': 73, 'label': 'LOCATION', 'text': 'West Laurenland General Hospital'}]


In [ ]:
PHI_LABELS = ["O"]

entity_types = [
    "NAME", "LOCATION", "DATE", "AGE", "PHONE",
    "EMAIL", "SSN", "MRN", "HEALTH_PLAN_ID",
    "ACCOUNT_ID", "LICENSE_ID", "URL", "IP"
]

for entity in entity_types:
    PHI_LABELS.extend([f"B-{entity}", f"I-{entity}"])

label2id = {label: i for i, label in enumerate(PHI_LABELS)}
id2label = {i: label for label, i in label2id.items()}

print("Total labels:", len(PHI_LABELS))
print("Labels restored successfully!")

Total labels: 27
Labels restored successfully!


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully!
Vocabulary size: 30522


In [ ]:
def align_example(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        return_offsets_mapping=True
    )

    labels = []

    for start, end in tokenized["offset_mapping"]:
        if start == end:
            labels.append(-100)
            continue

        tag = "O"

        for entity in example["annotations"]:
            if start < entity["end"] and end > entity["start"]:
                prefix = "B-" if start <= entity["start"] else "I-"
                tag = prefix + entity["label"]
                break

        labels.append(label2id[tag])

    tokenized.pop("offset_mapping")
    tokenized["labels"] = labels

    return tokenized

print("Token alignment function restored!")

Token alignment function restored!


In [ ]:
import torch
from torch.utils.data import Dataset

class PHIDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(item["labels"], dtype=torch.long)
        }

print("PyTorch dataset class restored!")

PyTorch dataset class restored!


In [ ]:
import numpy as np
from seqeval.metrics import classification_report

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        pred_seq = []
        label_seq = []

        for pred, actual in zip(prediction, label):
            if actual == -100:
                continue

            pred_seq.append(id2label[int(pred)])
            label_seq.append(id2label[int(actual)])

        true_predictions.append(pred_seq)
        true_labels.append(label_seq)

    report = classification_report(
        true_labels,
        true_predictions,
        output_dict=True,
        zero_division=0
    )

    return {
        "precision": report["micro avg"]["precision"],
        "recall": report["micro avg"]["recall"],
        "f1": report["micro avg"]["f1-score"]
    }

print("Evaluation metrics function restored!")

Evaluation metrics function restored!


In [ ]:
random.seed(42)
np.random.seed(42)

random.shuffle(combined_data)

aligned_combined_data = [
    align_example(example)
    for example in combined_data
]

print("Total tokenized training examples:", len(aligned_combined_data))
print("Independent test notes:", len(test_annotated_data))

Total tokenized training examples: 403
Independent test notes: 20


In [ ]:
from sklearn.model_selection import train_test_split

train_data, validation_data = train_test_split(
    aligned_combined_data,
    test_size=0.20,
    random_state=42
)

train_dataset = PHIDataset(train_data)
validation_dataset = PHIDataset(validation_data)

print("Training notes:", len(train_dataset))
print("Validation notes:", len(validation_dataset))

Training notes: 322
Validation notes: 81


In [ ]:
from transformers import DataCollatorForTokenClassification
from torch.utils.data import DataLoader

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=data_collator
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=data_collator
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("DataLoaders created successfully!")

Training batches: 41
Validation batches: 11
DataLoaders created successfully!


In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(PHI_LABELS),
    id2label=id2label,
    label2id=label2id
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Model loaded successfully!")
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!
Total parameters: 66,383,643
Trainable parameters: 66,383,643


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./phi_deid_improved",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Training configuration ready!")

Training configuration ready!


In [ ]:
training_results = trainer.train()

print("\nTraining completed successfully!")
print("Final training loss:", training_results.training_loss)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.736868,0.551069,0.872727,0.797342,0.833333
2,0.135187,0.087744,0.929487,0.963455,0.946166
3,0.050256,0.038399,0.976898,0.983389,0.980132
4,0.031686,0.029482,0.983444,0.986711,0.985075
5,0.024241,0.025671,0.983444,0.986711,0.985075


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed successfully!
Final training loss: 0.39660154988126056


In [ ]:
test_data = [
    align_example(example)
    for example in test_annotated_data
]

test_dataset = PHIDataset(test_data)

test_results = trainer.predict(test_dataset)

print("\nINDEPENDENT TEST RESULTS")
print("Test Precision:", test_results.metrics["test_precision"])
print("Test Recall:", test_results.metrics["test_recall"])
print("Test F1:", test_results.metrics["test_f1"])

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



INDEPENDENT TEST RESULTS
Test Precision: 0.9404761904761905
Test Recall: 0.9875
Test F1: 0.9634146341463415


In [ ]:
from seqeval.metrics import classification_report

predictions = np.argmax(test_results.predictions, axis=2)
labels = test_results.label_ids

true_predictions = []
true_labels = []

for prediction, label in zip(predictions, labels):
    pred_seq = []
    label_seq = []

    for pred, actual in zip(prediction, label):
        if actual == -100:
            continue

        pred_seq.append(id2label[int(pred)])
        label_seq.append(id2label[int(actual)])

    true_predictions.append(pred_seq)
    true_labels.append(label_seq)

print("PER-CATEGORY TEST PERFORMANCE")
print(classification_report(
    true_labels,
    true_predictions,
    zero_division=0
))

PER-CATEGORY TEST PERFORMANCE
              precision    recall  f1-score   support

         AGE       1.00      1.00      1.00         4
        DATE       1.00      1.00      1.00        16
       EMAIL       1.00      1.00      1.00         4
    LOCATION       0.83      1.00      0.91        20
         MRN       1.00      1.00      1.00         4
        NAME       0.96      0.96      0.96        24
       PHONE       1.00      1.00      1.00         8

   micro avg       0.94      0.99      0.96        80
   macro avg       0.97      0.99      0.98        80
weighted avg       0.95      0.99      0.96        80



In [ ]:
false_locations = []

for i, example in enumerate(test_annotated_data):
    encoded = tokenizer(
        example["text"],
        truncation=True,
        return_offsets_mapping=True
    )

    offsets = encoded["offset_mapping"]
    predicted_tags = predictions[i]

    predicted_spans = []
    current_start = None
    current_end = None

    for j, (start, end) in enumerate(offsets):
        if start == end:
            continue

        tag = id2label[int(predicted_tags[j])]

        if tag == "B-LOCATION":
            if current_start is not None:
                predicted_spans.append((current_start, current_end))
            current_start, current_end = start, end

        elif tag == "I-LOCATION" and current_start is not None:
            current_end = end

        else:
            if current_start is not None:
                predicted_spans.append((current_start, current_end))
                current_start = None
                current_end = None

    if current_start is not None:
        predicted_spans.append((current_start, current_end))

    actual_spans = [
        (entity["start"], entity["end"])
        for entity in example["annotations"]
        if entity["label"] == "LOCATION"
    ]

    for start, end in predicted_spans:
        if not any(start < gold_end and end > gold_start
                   for gold_start, gold_end in actual_spans):
            false_locations.append({
                "note": example["id"],
                "prediction": example["text"][start:end],
                "context": example["text"]
            })

print("Incorrect LOCATION predictions:", len(false_locations))

for item in false_locations:
    print("\nNote:", item["note"])
    print("Incorrect prediction:", item["prediction"])
    print("Full note:", item["context"])

Incorrect LOCATION predictions: 0


In [ ]:
for i, example in enumerate(test_annotated_data):
    encoded = tokenizer(
        example["text"],
        truncation=True,
        return_offsets_mapping=True
    )

    offsets = encoded["offset_mapping"]
    predicted_tags = predictions[i]

    actual_locations = [
        e["text"] for e in example["annotations"]
        if e["label"] == "LOCATION"
    ]

    predicted_locations = []
    current_start = None
    current_end = None

    for j, (start, end) in enumerate(offsets):
        if start == end:
            continue

        tag = id2label[int(predicted_tags[j])]

        if tag == "B-LOCATION":
            if current_start is not None:
                predicted_locations.append(
                    example["text"][current_start:current_end]
                )
            current_start, current_end = start, end

        elif tag == "I-LOCATION" and current_start is not None:
            current_end = end

        else:
            if current_start is not None:
                predicted_locations.append(
                    example["text"][current_start:current_end]
                )
                current_start = None
                current_end = None

    if current_start is not None:
        predicted_locations.append(
            example["text"][current_start:current_end]
        )

    if actual_locations != predicted_locations:
        print("\nNOTE:", example["id"])
        print("Actual:", actual_locations)
        print("Predicted:", predicted_locations)

In [ ]:
for i, example in enumerate(test_annotated_data):
    encoded = tokenizer(
        example["text"],
        truncation=True,
        return_offsets_mapping=True
    )

    offsets = encoded["offset_mapping"]
    predicted_tags = predictions[i]

    actual_locations = [
        e["text"] for e in example["annotations"]
        if e["label"] == "LOCATION"
    ]

    predicted_locations = []
    current_start = None
    current_end = None

    for j, (start, end) in enumerate(offsets):
        if start == end:
            continue

        tag = id2label[int(predicted_tags[j])]

        if tag == "B-LOCATION":
            if current_start is not None:
                predicted_locations.append(
                    example["text"][current_start:current_end]
                )
            current_start, current_end = start, end

        elif tag == "I-LOCATION" and current_start is not None:
            current_end = end

        else:
            if current_start is not None:
                predicted_locations.append(
                    example["text"][current_start:current_end]
                )
                current_start = None
                current_end = None

    if current_start is not None:
        predicted_locations.append(
            example["text"][current_start:current_end]
        )

    if actual_locations != predicted_locations:
        print("\nNOTE:", example["id"])
        print("Actual:", actual_locations)
        print("Predicted:", predicted_locations)

In [ ]:
def deidentify(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True
    )

    offsets = inputs.pop("offset_mapping")[0].tolist()

    model_inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    model.eval()

    with torch.no_grad():
        outputs = model(**model_inputs)

    predicted_ids = outputs.logits.argmax(dim=-1)[0].tolist()

    entities = []
    current_entity = None

    for (start, end), tag_id in zip(offsets, predicted_ids):
        if start == end:
            continue

        tag = id2label[tag_id]

        if tag.startswith("B-"):
            if current_entity:
                entities.append(current_entity)

            current_entity = {
                "start": start,
                "end": end,
                "label": tag[2:]
            }

        elif tag.startswith("I-") and current_entity:
            if current_entity["label"] == tag[2:]:
                current_entity["end"] = end
            else:
                entities.append(current_entity)
                current_entity = None

        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None

    if current_entity:
        entities.append(current_entity)

    entities.sort(key=lambda item: item["start"])

    mapping = {}
    counters = {}
    replacements = []

    for entity in entities:
        label = entity["label"]
        original = text[entity["start"]:entity["end"]]

        key = (label, original)

        if key not in mapping:
            counters[label] = counters.get(label, 0) + 1
            mapping[key] = f"[{label}_{counters[label]}]"

        replacements.append((
            entity["start"],
            entity["end"],
            mapping[key]
        ))

    masked_text = text

    for start, end, placeholder in reversed(replacements):
        masked_text = masked_text[:start] + placeholder + masked_text[end:]

    return masked_text, mapping

print("De-identification function created successfully!")

De-identification function created successfully!


In [ ]:
sample_note = """
Patient John Miller visited Salem City Hospital on 12/03/2025.
Phone: 9876543210.
Email: john.miller@example.com.
"""

masked_text, mapping = deidentify(sample_note)

print("ORIGINAL NOTE:")
print(sample_note)

print("\nDE-IDENTIFIED NOTE:")
print(masked_text)

print("\nPRIVATE MAPPING:")
print(mapping)

ORIGINAL NOTE:

Patient John Miller visited Salem City Hospital on 12/03/2025.
Phone: 9876543210.
Email: john.miller@example.com.


DE-IDENTIFIED NOTE:

Patient [NAME_1] visited [LOCATION_1] on [DATE_1].
Phone: [PHONE_1].
Email: [EMAIL_1].


PRIVATE MAPPING:
{('NAME', 'John Miller'): '[NAME_1]', ('LOCATION', 'Salem City Hospital'): '[LOCATION_1]', ('DATE', '12/03/2025'): '[DATE_1]', ('PHONE', '9876543210'): '[PHONE_1]', ('EMAIL', 'john.miller@example.com'): '[EMAIL_1]'}


In [ ]:
def rehydrate(response, mapping):
    restored_text = response

    for (label, original), placeholder in sorted(
        mapping.items(),
        key=lambda item: len(item[1]),
        reverse=True
    ):
        restored_text = restored_text.replace(
            placeholder,
            original
        )

    return restored_text

print("Rehydration function created successfully!")

Rehydration function created successfully!


In [ ]:
restored_text = rehydrate(masked_text, mapping)

print("ORIGINAL NOTE:")
print(sample_note)

print("\nMASKED NOTE:")
print(masked_text)

print("\nRESTORED NOTE:")
print(restored_text)

print("\nROUND-TRIP SUCCESS:", restored_text == sample_note)

ORIGINAL NOTE:

Patient John Miller visited Salem City Hospital on 12/03/2025.
Phone: 9876543210.
Email: john.miller@example.com.


MASKED NOTE:

Patient [NAME_1] visited [LOCATION_1] on [DATE_1].
Phone: [PHONE_1].
Email: [EMAIL_1].


RESTORED NOTE:

Patient John Miller visited Salem City Hospital on 12/03/2025.
Phone: 9876543210.
Email: john.miller@example.com.


ROUND-TRIP SUCCESS: True


In [ ]:
repeat_note = """
John Miller visited Salem Hospital.
John Miller returned on 15/03/2025.
Contact John Miller at 9876543210.
"""

repeat_masked, repeat_mapping = deidentify(repeat_note)

print("MASKED NOTE:")
print(repeat_masked)

print("\nMAPPING:")
print(repeat_mapping)~

SyntaxError: invalid syntax (4111436544.py, line 13)

In [ ]:
repeat_note = """
John Miller visited Salem Hospital.
John Miller returned on 15/03/2025.
Contact John Miller at 9876543210.
"""

repeat_masked, repeat_mapping = deidentify(repeat_note)

print("MASKED NOTE:")
print(repeat_masked)

print("\nMAPPING:")
print(repeat_mapping)


MASKED NOTE:

[NAME_1] visited [LOCATION_1].
[NAME_1] returned on [DATE_1].
Contact [NAME_1] at [PHONE_1].


MAPPING:
{('NAME', 'John Miller'): '[NAME_1]', ('LOCATION', 'Salem Hospital'): '[LOCATION_1]', ('DATE', '15/03/2025'): '[DATE_1]', ('PHONE', '9876543210'): '[PHONE_1]'}


In [ ]:
!pip install presidio-analyzer presidio-anonymizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 25.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 23.3 MB/s  0:00:00
  Attempting uninstall: preshed
    Found existing installation: preshed 3.0.15
    Uninstalling preshed-3.0.15:
      Successfully uninstalled preshed-3.0.15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [presidio-analyzer]


In [ ]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 24.8 MB/s  0:00:13
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
from presidio_analyzer import AnalyzerEngine

nlp = spacy.load("en_core_web_lg")
analyzer = AnalyzerEngine()

print("spaCy model loaded successfully!")
print("Presidio Analyzer initialized successfully!")


spaCy model loaded successfully!
Presidio Analyzer initialized successfully!


In [ ]:
presidio_results = analyzer.analyze(
    text=sample_note,
    language="en"
)

for result in presidio_results:
    print(
        "Entity:", result.entity_type,
        "| Text:", sample_note[result.start:result.end],
        "| Confidence:", round(result.score, 2)
    )

Entity: UK_NHS | Text: 9876543210 | Confidence: 1.0
Entity: EMAIL_ADDRESS | Text: john.miller@example.com | Confidence: 1.0
Entity: PERSON | Text: John Miller | Confidence: 0.85
Entity: DATE_TIME | Text: 12/03/2025 | Confidence: 0.85
Entity: PHONE_NUMBER | Text: 9876543210 | Confidence: 0.75
Entity: URL | Text: john.mil | Confidence: 0.5
Entity: URL | Text: example.com | Confidence: 0.5
Entity: US_BANK_NUMBER | Text: 9876543210 | Confidence: 0.05
Entity: US_DRIVER_LICENSE | Text: 9876543210 | Confidence: 0.01


In [ ]:
from collections import Counter

gold_counts = Counter()

for example in test_annotated_data:
    for entity in example["annotations"]:
        gold_counts[entity["label"]] += 1

print("TEST SET — GROUND-TRUTH ENTITY COUNTS")
print("=" * 45)

for label, count in sorted(gold_counts.items()):
    print(f"{label}: {count}")

print("\nTotal entities:", sum(gold_counts.values()))
print("Total test notes:", len(test_annotated_data))

TEST SET — GROUND-TRUTH ENTITY COUNTS
AGE: 4
DATE: 16
EMAIL: 4
LOCATION: 20
MRN: 4
NAME: 24
PHONE: 8

Total entities: 80
Total test notes: 20


In [ ]:
presidio_label_map = {
    "PERSON": "NAME",
    "LOCATION": "LOCATION",
    "DATE_TIME": "DATE",
    "PHONE_NUMBER": "PHONE",
    "UK_NHS": "PHONE",
    "EMAIL_ADDRESS": "EMAIL",
    "US_SSN": "SSN",
    "US_DRIVER_LICENSE": "LICENSE_ID",
    "URL": "URL",
    "IP_ADDRESS": "IP"
}

print("Presidio entity mapping:")
for presidio_label, project_label in presidio_label_map.items():
    print(f"{presidio_label:20} -> {project_label}")

Presidio entity mapping:
PERSON               -> NAME
LOCATION             -> LOCATION
DATE_TIME            -> DATE
PHONE_NUMBER         -> PHONE
UK_NHS               -> PHONE
EMAIL_ADDRESS        -> EMAIL
US_SSN               -> SSN
US_DRIVER_LICENSE    -> LICENSE_ID
URL                  -> URL
IP_ADDRESS           -> IP


In [ ]:
presidio_predictions = []

for example in test_annotated_data:
    results = analyzer.analyze(
        text=example["text"],
        language="en"
    )

    note_predictions = []

    for result in results:
        if result.entity_type in presidio_label_map:
            note_predictions.append({
                "start": result.start,
                "end": result.end,
                "label": presidio_label_map[result.entity_type],
                "text": example["text"][result.start:result.end],
                "confidence": result.score
            })

    presidio_predictions.append(note_predictions)

print("Presidio analysis completed!")
print("Test notes processed:", len(presidio_predictions))
print("Total mapped predictions:", sum(len(x) for x in presidio_predictions))

print("\nFirst test note predictions:")
for item in presidio_predictions[0]:
    print(item)

Presidio analysis completed!
Test notes processed: 20
Total mapped predictions: 83

First test note predictions:
{'start': 12, 'end': 24, 'label': 'NAME', 'text': 'Tiffany Lane', 'confidence': 0.85}
{'start': 77, 'end': 87, 'label': 'DATE', 'text': '11/01/1971', 'confidence': 0.6}
{'start': 96, 'end': 106, 'label': 'LICENSE_ID', 'text': '5364015794', 'confidence': 0.01}


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

gold_entities = []
predicted_entities = []

for example, predictions in zip(test_annotated_data, presidio_predictions):
    gold = {
        (e["start"], e["end"], e["label"])
        for e in example["annotations"]
    }

    predicted = {
        (e["start"], e["end"], e["label"])
        for e in predictions
    }

    gold_entities.extend(gold)
    predicted_entities.extend(predicted)

correct = len(set(gold_entities) & set(predicted_entities))
total_gold = len(set(gold_entities))
total_predicted = len(set(predicted_entities))

precision = correct / total_predicted if total_predicted else 0
recall = correct / total_gold if total_gold else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

print("PRESIDIO BASELINE — EXACT ENTITY MATCH")
print("=" * 45)
print("Correct predictions:", correct)
print("Total actual entities:", total_gold)
print("Total predictions:", total_predicted)
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-score:  {f1:.4f} ({f1*100:.2f}%)")

PRESIDIO BASELINE — EXACT ENTITY MATCH
Correct predictions: 44
Total actual entities: 69
Total predictions: 70
Precision: 0.6286 (62.86%)
Recall:    0.6377 (63.77%)
F1-score:  0.6331 (63.31%)


In [ ]:
gold_entities = []
predicted_entities = []

for note_id, (example, predictions) in enumerate(
    zip(test_annotated_data, presidio_predictions)
):
    for e in example["annotations"]:
        gold_entities.append(
            (note_id, e["start"], e["end"], e["label"])
        )

    for e in predictions:
        predicted_entities.append(
            (note_id, e["start"], e["end"], e["label"])
        )

gold_set = set(gold_entities)
predicted_set = set(predicted_entities)

correct = len(gold_set & predicted_set)
total_gold = len(gold_set)
total_predicted = len(predicted_set)

precision = correct / total_predicted if total_predicted else 0
recall = correct / total_gold if total_gold else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

print("CORRECTED PRESIDIO BASELINE")
print("=" * 40)
print("Actual entities:", total_gold)
print("Predicted entities:", total_predicted)
print("Correct exact matches:", correct)
print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print(f"F1-score:  {f1*100:.2f}%")

CORRECTED PRESIDIO BASELINE
Actual entities: 80
Predicted entities: 82
Correct exact matches: 53
Precision: 64.63%
Recall:    66.25%
F1-score:  65.43%


In [ ]:
custom_predictions = []

for example in test_annotated_data:
    masked_text, mapping = deidentify(example["text"])

    note_predictions = []

    for (label, original), placeholder in mapping.items():
        start = example["text"].find(original)

        if start != -1:
            note_predictions.append({
                "start": start,
                "end": start + len(original),
                "label": label,
                "text": original
            })

    custom_predictions.append(note_predictions)

print("Custom model evaluation predictions created!")
print("Test notes:", len(custom_predictions))
print("Total predictions:", sum(len(x) for x in custom_predictions))
print("\nFirst note:")
for item in custom_predictions[0]:
    print(item)

Custom model evaluation predictions created!
Test notes: 20
Total predictions: 79

First note:
{'start': 12, 'end': 24, 'label': 'NAME', 'text': 'Tiffany Lane'}
{'start': 41, 'end': 73, 'label': 'LOCATION', 'text': 'West Laurenland General Hospital'}
{'start': 77, 'end': 87, 'label': 'DATE', 'text': '11/01/1971'}
{'start': 96, 'end': 106, 'label': 'PHONE', 'text': '5364015794'}


In [ ]:
gold_entities = []
predicted_entities = []

for note_id, (example, predictions) in enumerate(
    zip(test_annotated_data, custom_predictions)
):
    for e in example["annotations"]:
        gold_entities.append(
            (note_id, e["start"], e["end"], e["label"])
        )

    for e in predictions:
        predicted_entities.append(
            (note_id, e["start"], e["end"], e["label"])
        )

gold_set = set(gold_entities)
predicted_set = set(predicted_entities)

correct = len(gold_set & predicted_set)
total_gold = len(gold_set)
total_predicted = len(predicted_set)

precision = correct / total_predicted if total_predicted else 0
recall = correct / total_gold if total_gold else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

print("CUSTOM MODEL — EXACT ENTITY MATCH")
print("=" * 45)
print("Actual entities:", total_gold)
print("Predicted entities:", total_predicted)
print("Correct exact matches:", correct)
print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print(f"F1-score:  {f1*100:.2f}%")

CUSTOM MODEL — EXACT ENTITY MATCH
Actual entities: 80
Predicted entities: 79
Correct exact matches: 79
Precision: 100.00%
Recall:    98.75%
F1-score:  99.37%


In [ ]:
def calculate_leak_rate(test_data, predictions):
    leaked_notes = 0
    total_missed = 0

    for note_id, (example, note_predictions) in enumerate(
        zip(test_data, predictions)
    ):
        predicted = {
            (p["start"], p["end"], p["label"])
            for p in note_predictions
        }

        missed = [
            e for e in example["annotations"]
            if (e["start"], e["end"], e["label"]) not in predicted
        ]

        if missed:
            leaked_notes += 1
            total_missed += len(missed)

    leak_rate = leaked_notes / len(test_data) * 100

    return leaked_notes, total_missed, leak_rate


p_leaked, p_missed, p_rate = calculate_leak_rate(
    test_annotated_data, presidio_predictions
)

c_leaked, c_missed, c_rate = calculate_leak_rate(
    test_annotated_data, custom_predictions
)

print("IDENTIFIER LEAK RATE COMPARISON")
print("=" * 45)
print(f"{'Metric':<25}{'Presidio':>10}{'Custom':>10}")
print("-" * 45)
print(f"{'Notes with leaks':<25}{p_leaked:>10}{c_leaked:>10}")
print(f"{'Missed identifiers':<25}{p_missed:>10}{c_missed:>10}")
print(f"{'Leak rate':<25}{p_rate:>9.2f}%{c_rate:>9.2f}%")


IDENTIFIER LEAK RATE COMPARISON
Metric                     Presidio    Custom
---------------------------------------------
Notes with leaks                 18         1
Missed identifiers               27         1
Leak rate                    90.00%     5.00%


In [ ]:
print("MISSED IDENTIFIERS — CUSTOM MODEL")
print("=" * 45)

for note_id, (example, predictions) in enumerate(
    zip(test_annotated_data, custom_predictions)
):
    predicted = {
        (p["start"], p["end"], p["label"])
        for p in predictions
    }

    for entity in example["annotations"]:
        key = (
            entity["start"],
            entity["end"],
            entity["label"]
        )

        if key not in predicted:
            print(f"\nTest note: {note_id + 1}")
            print("Missed entity:", entity)
            print("Clinical note:", example["text"])

MISSED IDENTIFIERS — CUSTOM MODEL

Test note: 18
Missed entity: {'start': 0, 'end': 16, 'label': 'NAME', 'text': 'Mr. Casey Campos'}
Clinical note: Mr. Casey Campos contacted the clinic from Marytown. Email: dixonmichael@example.com. Appointment date: 06/12/2010.


In [ ]:
note_18 = test_annotated_data[17]["text"]

masked_18, mapping_18 = deidentify(note_18)

print("ORIGINAL NOTE:")
print(note_18)

print("\nMASKED NOTE:")
print(masked_18)

print("\nDETECTED IDENTIFIERS:")
for key, placeholder in mapping_18.items():
    print(repr(key[1]), "->", placeholder)

ORIGINAL NOTE:
Mr. Casey Campos contacted the clinic from Marytown. Email: dixonmichael@example.com. Appointment date: 06/12/2010.

MASKED NOTE:
Mr. Casey Campos contacted the clinic from [LOCATION_1]. Email: [EMAIL_1]. Appointment date: [DATE_1].

DETECTED IDENTIFIERS:
'Marytown' -> [LOCATION_1]
'dixonmichael@example.com' -> [EMAIL_1]
'06/12/2010' -> [DATE_1]


In [ ]:
note_18 = test_annotated_data[17]["text"]

masked_18, mapping_18 = deidentify(note_18)

print("ORIGINAL NOTE:")
print(note_18)

print("\nMASKED NOTE:")
print(masked_18)

print("\nDETECTED IDENTIFIERS:")
for key, placeholder in mapping_18.items():
    print(repr(key[1]), "->", placeholder)

ORIGINAL NOTE:
Mr. Casey Campos contacted the clinic from Marytown. Email: dixonmichael@example.com. Appointment date: 06/12/2010.

MASKED NOTE:
Mr. Casey Campos contacted the clinic from [LOCATION_1]. Email: [EMAIL_1]. Appointment date: [DATE_1].

DETECTED IDENTIFIERS:
'Marytown' -> [LOCATION_1]
'dixonmichael@example.com' -> [EMAIL_1]
'06/12/2010' -> [DATE_1]


In [ ]:
import re

def add_title_based_names(text, entities):
    pattern = r"\b(?:Mr|Mrs|Ms|Miss|Dr)\.?\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*"

    existing = {
        (e["start"], e["end"])
        for e in entities
    }

    for match in re.finditer(pattern, text):
        span = (match.start(), match.end())

        if span not in existing:
            entities.append({
                "start": match.start(),
                "end": match.end(),
                "label": "NAME"
            })

    return sorted(entities, key=lambda e: e["start"])

print("Title-based name detection rule added!")

Title-based name detection rule added!


In [ ]:
note_18_entities = add_title_based_names(
    note_18,
    []
)

print("DETECTED TITLE-BASED NAMES")

for entity in note_18_entities:
    print(
        note_18[entity["start"]:entity["end"]],
        "->",
        entity["label"]
    )

DETECTED TITLE-BASED NAMES
Mr. Casey Campos -> NAME


In [ ]:
import re

original_deidentify = deidentify

def deidentify(text):
    masked_text, mapping = original_deidentify(text)

    title_pattern = r"\b(?:Mr|Mrs|Ms|Miss|Dr)\.?\s+[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*"

    title_matches = list(re.finditer(title_pattern, text))

    for match in reversed(title_matches):
        original_name = match.group()
        key = ("NAME", original_name)

        if key not in mapping:
            name_count = sum(
                1 for label, _ in mapping if label == "NAME"
            ) + 1

            placeholder = f"[NAME_{name_count}]"
            mapping[key] = placeholder

        else:
            placeholder = mapping[key]

        masked_text = (
            masked_text[:match.start()]
            + placeholder
            + masked_text[match.end():]
        )

    return masked_text, mapping

print("Title-based name detection integrated into gateway!")

Title-based name detection integrated into gateway!


In [ ]:
masked_18, mapping_18 = deidentify(note_18)

print("ORIGINAL NOTE:")
print(note_18)

print("\nCORRECTED MASKED NOTE:")
print(masked_18)

print("\nMAPPING:")
for key, value in mapping_18.items():
    print(repr(key[1]), "->", value)

restored_18 = rehydrate(masked_18, mapping_18)

print("\nROUND-TRIP SUCCESS:", restored_18 == note_18)

ORIGINAL NOTE:
Mr. Casey Campos contacted the clinic from Marytown. Email: dixonmichael@example.com. Appointment date: 06/12/2010.

CORRECTED MASKED NOTE:
[NAME_1] contacted the clinic from [LOCATION_1]. Email: [EMAIL_1]. Appointment date: [DATE_1].

MAPPING:
'Marytown' -> [LOCATION_1]
'dixonmichael@example.com' -> [EMAIL_1]
'06/12/2010' -> [DATE_1]
'Mr. Casey Campos' -> [NAME_1]

ROUND-TRIP SUCCESS: True


In [ ]:
age_test_notes = [
    "Patient John Smith is 45 years old.",
    "Patient David Brown is 89 years old.",
    "Patient Robert Jones is 90 years old.",
    "Patient William Davis is 95 years old."
]

for note in age_test_notes:
    masked, mapping = deidentify(note)

    print("\nOriginal:", note)
    print("Masked:  ", masked)


Original: Patient John Smith is 45 years old.
Masked:   Patient [NAME_1] is [AGE_1] years old.

Original: Patient David Brown is 89 years old.
Masked:   Patient [NAME_1] is [AGE_1] years old.

Original: Patient Robert Jones is 90 years old.
Masked:   Patient [NAME_1] is [AGE_1] years old.

Original: Patient William Davis is 95 years old.
Masked:   Patient [NAME_1] is [AGE_1] years old.


In [ ]:
import re
import torch

def extract_phi_entities(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        return_offsets_mapping=True,
        truncation=True,
        max_length=512
    )

    offsets = inputs.pop("offset_mapping")[0].tolist()

    model_inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    model.eval()

    with torch.no_grad():
        outputs = model(**model_inputs)

    predicted_ids = outputs.logits.argmax(dim=-1)[0].tolist()

    entities = []
    current_entity = None

    for (start, end), tag_id in zip(offsets, predicted_ids):

        if start == end:
            continue

        tag = id2label[tag_id]

        if tag.startswith("B-"):

            if current_entity:
                entities.append(current_entity)

            current_entity = {
                "start": start,
                "end": end,
                "label": tag[2:]
            }

        elif tag.startswith("I-") and current_entity:

            if current_entity["label"] == tag[2:]:
                current_entity["end"] = end
            else:
                entities.append(current_entity)
                current_entity = None

        else:

            if current_entity:
                entities.append(current_entity)
                current_entity = None

    if current_entity:
        entities.append(current_entity)

    # Correct age policy: only mask ages above 89
    for match in re.finditer(
        r"\b(?:age[d]?\s*[:=]?\s*|is\s+)(\d{1,3})(?=\s*(?:years?\s+old|year-old|y/o))",
        text,
        re.IGNORECASE
    ):

        age = int(match.group(1))

        if age > 89:
            entities.append({
                "start": match.start(1),
                "end": match.end(1),
                "label": "AGE"
            })

    # Detect names with titles
    title_pattern = (
        r"\b(?:Mr|Mrs|Ms|Miss|Dr)\.?\s+"
        r"[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*"
    )

    title_entities = []

    for match in re.finditer(title_pattern, text):

        title_entities.append({
            "start": match.start(),
            "end": match.end(),
            "label": "NAME",
            "priority": 2
        })

    # Remove model entities overlapping title-based names
    for title_entity in title_entities:

        entities = [
            e for e in entities
            if not (
                e["start"] < title_entity["end"]
                and e["end"] > title_entity["start"]
            )
        ]

        entities.append(title_entity)

    # Preserve relative time expressions
    relative_pattern = re.compile(
        r"\b\d+\s+(?:seconds?|minutes?|hours?|days?|weeks?|months?|years?)"
        r"\s+(?:post[- ]?op|after|before|prior|later|ago)\b",
        re.IGNORECASE
    )

    relative_spans = [
        (m.start(), m.end())
        for m in relative_pattern.finditer(text)
    ]

    filtered_entities = []

    for entity in entities:

        start = entity["start"]
        end = entity["end"]
        label = entity["label"]

        original = text[start:end]

        # Never mask ages 89 or below
        if label == "AGE":
            try:
                if int(original) <= 89:
                    continue
            except ValueError:
                continue

        # Do not mask relative time expressions
        if label == "DATE":
            if any(
                start < relative_end and end > relative_start
                for relative_start, relative_end in relative_spans
            ):
                continue

        # Protect Parkinson's disease and similar disease phrases
        if label == "NAME":
            following_text = text[end:end + 30]

            if re.match(
                r"['’]s\s+disease\b",
                following_text,
                re.IGNORECASE
            ):
                continue

        filtered_entities.append({
            "start": start,
            "end": end,
            "label": label
        })

    # Remove overlapping duplicate predictions
    filtered_entities.sort(
        key=lambda e: (
            -e.get("priority", 0),
            e["start"],
            -(e["end"] - e["start"])
        )
    )

    final_entities = []

    for entity in filtered_entities:

        overlaps = any(
            entity["start"] < existing["end"]
            and entity["end"] > existing["start"]
            for existing in final_entities
        )

        if not overlaps:
            final_entities.append(entity)

    return sorted(final_entities, key=lambda e: e["start"])


print("Final entity detection function created!")

Final entity detection function created!


In [ ]:
def deidentify(text):

    entities = extract_phi_entities(text)

    mapping = {}
    counters = {}
    replacements = []

    for entity in entities:

        start = entity["start"]
        end = entity["end"]
        label = entity["label"]

        original = text[start:end]

        key = (label, original)

        if key not in mapping:

            counters[label] = counters.get(label, 0) + 1

            placeholder = f"[{label}_{counters[label]}]"

            mapping[key] = placeholder

        else:
            placeholder = mapping[key]

        replacements.append(
            (start, end, placeholder)
        )

    masked_text = text

    for start, end, placeholder in reversed(replacements):

        masked_text = (
            masked_text[:start]
            + placeholder
            + masked_text[end:]
        )

    return masked_text, mapping


print("Final deidentify() function created!")

Final deidentify() function created!


In [ ]:
def rehydrate(response, mapping):

    restored_text = response

    for (label, original), placeholder in sorted(
        mapping.items(),
        key=lambda item: len(item[1]),
        reverse=True
    ):

        restored_text = restored_text.replace(
            placeholder,
            original
        )

    return restored_text


print("Final rehydrate() function created!")

Final rehydrate() function created!


In [ ]:
final_test_note = """
Mr. Casey Campos visited Salem City Hospital.
He is 95 years old.
Appointment date: 12/03/2025.
Contact: 9876543210.
He underwent surgery 3 days post-op.
"""

masked_note, identifier_mapping = deidentify(final_test_note)

print("ORIGINAL NOTE:")
print(final_test_note)

print("\nMASKED NOTE:")
print(masked_note)

print("\nIDENTIFIER MAPPING:")

for key, value in identifier_mapping.items():
    print(repr(key[1]), "->", value)

restored_note = rehydrate(
    masked_note,
    identifier_mapping
)

print("\nRESTORED NOTE:")
print(restored_note)

print(
    "\nROUND TRIP:",
    restored_note == final_test_note
)

ORIGINAL NOTE:

Mr. Casey Campos visited Salem City Hospital.
He is 95 years old.
Appointment date: 12/03/2025.
Contact: 9876543210.
He underwent surgery 3 days post-op.


MASKED NOTE:

[NAME_1] visited [LOCATION_1].
He is [AGE_1] years old.
Appointment date: [DATE_1].
Contact: [PHONE_1].
He underwent surgery 3 days post-op.


IDENTIFIER MAPPING:
'Mr. Casey Campos' -> [NAME_1]
'Salem City Hospital' -> [LOCATION_1]
'95' -> [AGE_1]
'12/03/2025' -> [DATE_1]
'9876543210' -> [PHONE_1]

RESTORED NOTE:

Mr. Casey Campos visited Salem City Hospital.
He is 95 years old.
Appointment date: 12/03/2025.
Contact: 9876543210.
He underwent surgery 3 days post-op.


ROUND TRIP: True


In [ ]:
special_test_notes = [
    "Patient John Smith is 45 years old.",
    "Patient David Brown is 89 years old.",
    "Patient Robert Jones is 90 years old.",
    "Patient William Davis is 95 years old.",
    "Dr. Parkinson diagnosed Parkinson's disease.",
    "Patient returned 3 days post-op.",
    "Mr. Casey Campos visited Salem Hospital."
]

for note in special_test_notes:

    masked, mapping = deidentify(note)

    print("\nORIGINAL:", note)
    print("MASKED:  ", masked)


ORIGINAL: Patient John Smith is 45 years old.
MASKED:   Patient [NAME_1] is 45 years old.

ORIGINAL: Patient David Brown is 89 years old.
MASKED:   Patient [NAME_1] is 89 years old.

ORIGINAL: Patient Robert Jones is 90 years old.
MASKED:   Patient [NAME_1] is [AGE_1] years old.

ORIGINAL: Patient William Davis is 95 years old.
MASKED:   Patient [NAME_1] is [AGE_1] years old.

ORIGINAL: Dr. Parkinson diagnosed Parkinson's disease.
MASKED:   [NAME_1] diagnosed Parkinson's disease.

ORIGINAL: Patient returned 3 days post-op.
MASKED:   Patient returned 3 days post-op.

ORIGINAL: Mr. Casey Campos visited Salem Hospital.
MASKED:   [NAME_1] visited [LOCATION_1].


In [ ]:
repeat_test = """
John Miller visited Salem Hospital.
John Miller returned on 15/03/2025.
Contact John Miller at 9876543210.
"""

repeat_masked, repeat_mapping = deidentify(repeat_test)

print("ORIGINAL:")
print(repeat_test)

print("\nMASKED:")
print(repeat_masked)

print("\nMAPPING:")
for key, value in repeat_mapping.items():
    print(key, "->", value)

print(
    "\nROUND TRIP:",
    rehydrate(repeat_masked, repeat_mapping) == repeat_test
)

ORIGINAL:

John Miller visited Salem Hospital.
John Miller returned on 15/03/2025.
Contact John Miller at 9876543210.


MASKED:

[NAME_1] visited [LOCATION_1].
[NAME_1] returned on [DATE_1].
Contact [NAME_1] at [PHONE_1].


MAPPING:
('NAME', 'John Miller') -> [NAME_1]
('LOCATION', 'Salem Hospital') -> [LOCATION_1]
('DATE', '15/03/2025') -> [DATE_1]
('PHONE', '9876543210') -> [PHONE_1]

ROUND TRIP: True


In [ ]:
custom_predictions = []

for example in test_annotated_data:

    entities = extract_phi_entities(example["text"])

    note_predictions = []

    for entity in entities:

        start = entity["start"]
        end = entity["end"]

        note_predictions.append({
            "start": start,
            "end": end,
            "label": entity["label"],
            "text": example["text"][start:end]
        })

    custom_predictions.append(note_predictions)

print("Final custom predictions generated!")

print("Test notes:", len(custom_predictions))

print(
    "Total predictions:",
    sum(len(x) for x in custom_predictions)
)

Final custom predictions generated!
Test notes: 20
Total predictions: 80


In [ ]:
gold_entities = []
predicted_entities = []

for note_id, (example, predictions) in enumerate(
    zip(test_annotated_data, custom_predictions)
):

    for entity in example["annotations"]:

        gold_entities.append((
            note_id,
            entity["start"],
            entity["end"],
            entity["label"]
        ))

    for entity in predictions:

        predicted_entities.append((
            note_id,
            entity["start"],
            entity["end"],
            entity["label"]
        ))

gold_set = set(gold_entities)
predicted_set = set(predicted_entities)

correct = len(gold_set & predicted_set)

total_gold = len(gold_set)
total_predicted = len(predicted_set)

precision = correct / total_predicted if total_predicted else 0

recall = correct / total_gold if total_gold else 0

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

print("FINAL CUSTOM MODEL RESULTS")
print("=" * 45)

print("Actual entities:", total_gold)
print("Predicted entities:", total_predicted)
print("Correct exact matches:", correct)

print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1-score:  {f1 * 100:.2f}%")

FINAL CUSTOM MODEL RESULTS
Actual entities: 80
Predicted entities: 80
Correct exact matches: 76
Precision: 95.00%
Recall:    95.00%
F1-score:  95.00%


In [ ]:
def calculate_leak_rate(test_data, predictions):

    leaked_notes = 0
    total_missed = 0

    for example, note_predictions in zip(
        test_data, predictions
    ):

        predicted = {
            (p["start"], p["end"], p["label"])
            for p in note_predictions
        }

        missed = [
            e for e in example["annotations"]
            if (
                e["start"],
                e["end"],
                e["label"]
            ) not in predicted
        ]

        if missed:
            leaked_notes += 1
            total_missed += len(missed)

    rate = leaked_notes / len(test_data) * 100

    return leaked_notes, total_missed, rate


p_leaked, p_missed, p_rate = calculate_leak_rate(
    test_annotated_data,
    presidio_predictions
)

c_leaked, c_missed, c_rate = calculate_leak_rate(
    test_annotated_data,
    custom_predictions
)

print("FINAL LEAK RATE COMPARISON")
print("=" * 45)

print(f"Presidio notes with leaks: {p_leaked}")
print(f"Custom notes with leaks:   {c_leaked}")

print(f"Presidio missed entities: {p_missed}")
print(f"Custom missed entities:   {c_missed}")

print(f"Presidio leak rate: {p_rate:.2f}%")
print(f"Custom leak rate:   {c_rate:.2f}%")

FINAL LEAK RATE COMPARISON
Presidio notes with leaks: 18
Custom notes with leaks:   4
Presidio missed entities: 27
Custom missed entities:   4
Presidio leak rate: 90.00%
Custom leak rate:   20.00%


In [ ]:
from collections import defaultdict

category_results = defaultdict(
    lambda: {"correct": 0, "predicted": 0, "actual": 0}
)

for note_id, (example, predictions) in enumerate(
    zip(test_annotated_data, custom_predictions)
):

    gold = {
        (e["start"], e["end"], e["label"])
        for e in example["annotations"]
    }

    predicted = {
        (e["start"], e["end"], e["label"])
        for e in predictions
    }

    for start, end, label in gold:
        category_results[label]["actual"] += 1

    for start, end, label in predicted:
        category_results[label]["predicted"] += 1

    for start, end, label in gold & predicted:
        category_results[label]["correct"] += 1


print("PER-CATEGORY EXACT MATCH PERFORMANCE")
print("=" * 75)

print(
    f"{'CATEGORY':<18}"
    f"{'PRECISION':>12}"
    f"{'RECALL':>12}"
    f"{'F1':>12}"
)

for label, values in sorted(category_results.items()):

    tp = values["correct"]
    pred = values["predicted"]
    actual = values["actual"]

    p = tp / pred if pred else 0
    r = tp / actual if actual else 0

    score = (
        2 * p * r / (p + r)
        if p + r else 0
    )

    print(
        f"{label:<18}"
        f"{p * 100:>11.2f}%"
        f"{r * 100:>11.2f}%"
        f"{score * 100:>11.2f}%"
    )

PER-CATEGORY EXACT MATCH PERFORMANCE
CATEGORY             PRECISION      RECALL          F1
AGE                    100.00%     100.00%     100.00%
DATE                   100.00%     100.00%     100.00%
EMAIL                  100.00%     100.00%     100.00%
LOCATION               100.00%     100.00%     100.00%
MRN                    100.00%     100.00%     100.00%
NAME                    83.33%      83.33%      83.33%
PHONE                  100.00%     100.00%     100.00%


In [ ]:
import os

save_directory = "./PHI_Deidentification_Final"

os.makedirs(save_directory, exist_ok=True)

model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print("Model and tokenizer saved successfully!")
print("Location:", save_directory)

print(
    "Total model parameters:",
    sum(p.numel() for p in model.parameters())
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!
Location: ./PHI_Deidentification_Final
Total model parameters: 66383643


In [ ]:
import os

for root, dirs, files in os.walk(save_directory):
    for file in files:
        print(os.path.join(root, file))


./PHI_Deidentification_Final/tokenizer.json
./PHI_Deidentification_Final/model.safetensors
./PHI_Deidentification_Final/tokenizer_config.json
./PHI_Deidentification_Final/config.json


In [ ]:
import shutil

zip_path = shutil.make_archive(
    "PHI_Deidentification_Final",
    "zip",
    root_dir=".",
    base_dir="PHI_Deidentification_Final"
)

print("ZIP created successfully!")
print("File:", zip_path)

ZIP created successfully!
File: /content/PHI_Deidentification_Final.zip


In [ ]:
from google.colab import files
files.download("PHI_Deidentification_Final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>